# 섹터 0 — MuJoCo 로 씬·오브젝트·로봇 구성하기

DIDEN_World 와 DIDEN_Core 는 둘 다 MuJoCo 위에서 돕니다. World 는 MuJoCo 모델을 GPU 로 올려 수천 개 환경을 병렬로 돌리고, Core 는 같은 로봇 모델을 실제 제어기와 묶어 돌립니다. 그래서 두 레포의 어떤 코드를 읽든 맨 밑바닥에는 **MJCF(XML) → `MjModel` → `MjData`** 가 있습니다. 이 섹터는 그 바닥을 직접 만들어 보는 곳입니다.

끝까지 풀면 다음을 손으로 만들 수 있게 됩니다.

| Part | 만드는 것 | 실제 코드에서 대응하는 곳 |
|---|---|---|
| 1 | MJCF 로 공 하나 떨어뜨리기, `MjModel` / `MjData` | 모든 것의 출발점 |
| 2 | 상태 벡터(`qpos`, `qvel`) 읽고 쓰기, 좌표계 | `DIDEN_World/packages/diden-common/src/diden_common/sim_adapter.py`, `math.py` |
| 3 | geom, 질량·관성, visual/collision, default class, include | `DIDEN_Asset/object/common.xml`, `DIDEN_Asset/robot/common/*.xml` |
| 4 | 접촉 필터, 마찰, 접촉력, 구르기 | `DIDEN_Asset/robot/diden_spider_v2/robot.xml` 의 `contype` / `conaffinity` / `condim` |
| 5 | 관절과 기구학 트리, Spider 다리 한 개, 정기구학 | `robot.xml` 의 `Leg_FL_*`, Core 의 기구학 |
| 6 | 액추에이터와 PD 제어 | `DIDEN_Asset/robot/common/actuator.xml`, `joint.xml`, World 의 `robots/diden_spider/cfg.py` |
| 7 | site 와 센서(IMU) | `robot.xml` 의 `Torso_IMU_site_IAM`, `accelerometer`, `gyro` |
| 8 | `MjSpec` 으로 씬 조립 — 섹터 1 에서 쓸 **공 굴리기 씬** | `DIDEN_World/packages/diden-engines/diden-mjlab/src/diden_mjlab/scene/scene.py` |
| 9 | 진짜 Spider 로봇 불러와 세우기 | `diden_mjlab/robots/diden_spider/spec.py` |

**푸는 방법**

- `# TODO` 가 있는 셀이 문제입니다. `raise NotImplementedError` 를 지우고 본문을 직접 칩니다.
- 바로 아래 채점 셀을 실행해 ✅ 가 나오면 다음으로 갑니다. ❌ 면 메시지를 읽고 고칩니다.
- "먼저 예상해 보기" 는 실행 전에 답을 적어 본 다음 실행해서 맞춰 봅니다. 틀린 예상이 가장 많이 남습니다.
- 30분 넘게 막히면 힌트(`<details>`)를 펴고, 그래도 안 되면 `s0_mujoco_scene_solutions.ipynb` 의 같은 셀을 봅니다.
- 이 노트북은 CPU 만 씁니다. GPU 병렬 시뮬레이션은 섹터 1 에서 시작합니다.


In [ ]:
import os
os.environ.setdefault("MUJOCO_GL", "egl")      # 창 없이 GPU 로 렌더링

from pathlib import Path
import numpy as np
import mujoco
import mediapy as media

np.set_printoptions(precision=4, suppress=True, linewidth=140)

# 레포 루트(= DIDEN_Asset 서브모듈이 있는 폴더)를 찾는다.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "DIDEN_Asset" / "object").is_dir())
ASSET = ROOT / "DIDEN_Asset"
print("mujoco", mujoco.__version__, "| asset:", ASSET)


In [ ]:
# 채점·렌더링 도우미 (주어진 코드)

def check(name, cond):
    """조건이 참이면 ✅, 아니면 ❌ 를 찍고 멈춘다."""
    print(("✅ " if cond else "❌ ") + name)
    assert cond, name


def check_close(name, got, want, atol=1e-6, rtol=0.0):
    got, want = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    ok = got.shape == want.shape and np.allclose(got, want, atol=atol, rtol=rtol)
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"\n   got  = {got}\n   want = {want}"))
    assert ok, name


def make_camera(lookat=(0, 0, 0.2), distance=3.0, azimuth=135, elevation=-25):
    cam = mujoco.MjvCamera()
    cam.lookat[:] = lookat
    cam.distance, cam.azimuth, cam.elevation = distance, azimuth, elevation
    return cam


def render(m, d, camera=None, width=480, height=360, groups=None):
    """현재 MjData 를 한 장 그린다. groups: 보이게 할 geom group 번호들 (기본은 MuJoCo 기본값 0~2)."""
    with mujoco.Renderer(m, height, width) as r:
        opt = mujoco.MjvOption()
        if groups is not None:
            opt.geomgroup[:] = 0
            for g in groups:
                opt.geomgroup[g] = 1
        r.update_scene(d, camera=camera if camera is not None else make_camera(), scene_option=opt)
        return r.render()


def record(m, d, seconds, camera=None, fps=30, step_fn=None, width=480, height=360, groups=None):
    """seconds 동안 시뮬레이션하며 프레임을 모은다. step_fn(m, d) 가 있으면 매 스텝 직전에 부른다."""
    frames, n_per_frame = [], max(1, round(1.0 / fps / m.opt.timestep))
    with mujoco.Renderer(m, height, width) as r:
        opt = mujoco.MjvOption()
        if groups is not None:
            opt.geomgroup[:] = 0
            for g in groups:
                opt.geomgroup[g] = 1
        cam = camera if camera is not None else make_camera()
        for i in range(round(seconds / m.opt.timestep)):
            if step_fn is not None:
                step_fn(m, d)
            mujoco.mj_step(m, d)
            if i % n_per_frame == 0:
                r.update_scene(d, camera=cam, scene_option=opt)
                frames.append(r.render())
    return frames


## Part 1 — 첫 번째 세계: 공 하나 떨어뜨리기

MuJoCo 를 쓰는 흐름은 항상 세 단계입니다.

```
MJCF (XML 문자열/파일)  ──컴파일──▶  MjModel (변하지 않는 것: 질량, 형상, 관절 구조, 옵션)
                                        │
                                        ▼
                                     MjData  (매 스텝 변하는 것: qpos, qvel, 접촉, 센서값)
                                        │  mj_step(model, data)
                                        ▼
                                     dt 만큼 지난 MjData
```

MJCF 의 뼈대는 이렇게 생겼습니다.

```xml
<mujoco model="이름">
  <option timestep="0.002"/>            <!-- 물리 설정 -->
  <worldbody>                            <!-- 세계 좌표계. 여기 바로 붙은 geom 은 움직이지 않는다 -->
    <light pos="0 0 3"/>
    <geom name="floor" type="plane" size="5 5 0.1"/>
    <body name="ball" pos="0 0 1">       <!-- body = 강체 하나 -->
      <freejoint name="ball_free"/>      <!-- 관절이 없으면 부모에 용접된다. freejoint = 6자유도 -->
      <geom name="ball_geom" type="sphere" size="0.1"/>   <!-- geom = 형상(충돌·질량·그림) -->
    </body>
  </worldbody>
</mujoco>
```

- **body** 는 좌표계 하나를 가진 강체입니다. 형상은 없습니다.
- **geom** 은 body 에 붙는 형상입니다. 충돌, 질량(밀도 × 부피), 렌더링이 모두 geom 에서 나옵니다.
- **joint** 는 body 가 **부모에 대해** 어떻게 움직일 수 있는지를 정합니다. joint 가 하나도 없는 body 는 부모에 고정됩니다.

### 문제 1-1. 낙하 씬 MJCF 쓰기

아래 조건의 MJCF 문자열을 돌려주는 함수를 씁니다.

| 항목 | 값 |
|---|---|
| timestep | 0.002 |
| 바닥 | `floor`, plane, size `5 5 0.1` |
| body | `ball`, 위치 `(0, 0, height)` |
| joint | freejoint `ball_free` |
| geom | `ball_geom`, sphere, 반지름 0.1 |
| 조명 | 아무 데나 하나 |


In [ ]:
def drop_xml(height: float) -> str:
    """높이 height 에서 공 하나가 떨어지는 MJCF 문자열."""
    return f"""
<mujoco model="drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3"/>
    <geom name="floor" type="plane" size="5 5 0.1"/>
    <body name="ball" pos="0 0 {height}">
      <freejoint name="ball_free"/>
      <geom name="ball_geom" type="sphere" size="0.1"/>
    </body>
  </worldbody>
</mujoco>"""


In [ ]:
_m = mujoco.MjModel.from_xml_string(drop_xml(1.0))
check("body 2개 (world + ball)", _m.nbody == 2)
check("geom 2개 (floor + ball_geom)", _m.ngeom == 2 and _m.geom("ball_geom").type[0] == mujoco.mjtGeom.mjGEOM_SPHERE)
check("timestep 0.002", _m.opt.timestep == 0.002)
check("공의 초기 위치", np.allclose(_m.body("ball").pos, [0, 0, 1.0]))
check("freejoint 이름", _m.joint("ball_free").type[0] == mujoco.mjtJoint.mjJNT_FREE)


### 먼저 예상해 보기

1. 이 모델의 `nq`(위치 좌표 개수)와 `nv`(속도 좌표 개수)는 각각 몇일까요? 둘이 왜 다를까요?
2. 공의 질량은 얼마일까요? MJCF 어디에도 질량을 쓰지 않았습니다.

적어 본 다음 아래 셀을 실행합니다.


In [ ]:
m = mujoco.MjModel.from_xml_string(drop_xml(1.0))
d = mujoco.MjData(m)
print("nq =", m.nq, " nv =", m.nv)
print("qpos0 =", m.qpos0, " ← [x y z | qw qx qy qz]")
print("ball mass =", m.body("ball").mass, " (기본 밀도 1000 kg/m³ × 구 부피)", 1000 * 4 / 3 * np.pi * 0.1**3)
media.show_image(render(m, d))


`nq = 7`, `nv = 6` 입니다. 자유 물체의 자세는 쿼터니언 4개로 적지만 각속도는 3개면 되기 때문입니다. 이 차이 때문에 MuJoCo 에서는 `qpos` 를 그냥 `qpos += qvel * dt` 로 적분하면 안 됩니다(`mj_integratePos` 가 따로 있는 이유).

### 문제 1-2. 시뮬레이션 돌리기

`mj_step(m, d)` 를 한 번 부르면 `d.time` 이 `m.opt.timestep` 만큼 흐릅니다. `seconds` 동안 돌리면서 매 스텝 **직후** 의 시간과 공의 높이를 모아 돌려줍니다.


In [ ]:
def simulate_height(m, d, seconds: float):
    """seconds 동안 스텝을 밟고 (times, heights) 를 numpy 배열로 돌려준다. 공의 높이는 qpos[2]."""
    n = round(seconds / m.opt.timestep)
    times, heights = [], []
    for _ in range(n):
        mujoco.mj_step(m, d)
        times.append(d.time)
        heights.append(d.qpos[2])
    return np.array(times), np.array(heights)


### 먼저 예상해 보기

공이 바닥에 닿기 전까지 높이는 $z(t) = h - \tfrac12 g t^2$ 를 **정확히** 따를까요? MuJoCo 의 기본 적분기는 반암시적 오일러(semi-implicit Euler)입니다.

$$v_{k+1} = v_k - g\,\Delta t, \qquad z_{k+1} = z_k + v_{k+1}\,\Delta t$$

이 점화식을 $n$ 번 풀면 $z_n$ 은 어떤 식이 될까요? 연속 시간 해와 얼마나 차이 날까요? 아래 채점 셀이 답을 확인합니다.


In [ ]:
m = mujoco.MjModel.from_xml_string(drop_xml(1.0))
d = mujoco.MjData(m)
_t, _z = simulate_height(m, d, 0.3)
_g, _dt = 9.81, m.opt.timestep
_n = np.arange(1, len(_t) + 1)
check("0.3초 = 150 스텝", len(_t) == 150 and abs(_t[-1] - 0.3) < 1e-9)
check_close("이산 해  z_n = h - g·dt²·n(n+1)/2 와 일치", _z, 1.0 - _g * _dt**2 * _n * (_n + 1) / 2, atol=1e-9)
print("   연속 해와의 차이(끝점):", _z[-1] - (1.0 - 0.5 * _g * _t[-1] ** 2), " ≈ -g·dt·t/2 =", -_g * _dt * _t[-1] / 2)
_t, _z = simulate_height(m, d, 2.0)
check("2초 뒤에는 바닥 위에 정지 (z ≈ 반지름)", abs(_z[-1] - 0.1) < 2e-3)
print("   정지 높이:", _z[-1], " → 반지름 0.1 보다 살짝 낮다. MuJoCo 의 접촉은 '부드러운' 접촉이라 조금 파고든다 (Part 4).")


In [ ]:
m = mujoco.MjModel.from_xml_string(drop_xml(1.0))
d = mujoco.MjData(m)
media.show_video(record(m, d, 1.5), fps=30)


**정리**

- `MjModel` 은 컴파일된 상수, `MjData` 는 상태입니다. World 가 환경 4096 개를 돌릴 때도 모델은 하나이고 데이터만 4096 벌입니다.
- 시뮬레이션 결과는 `timestep` 과 적분기에 따라 달라집니다. Spider 의 `robot.xml` 은 `timestep="0.0005" integrator="implicitfast"` 를 쓰고, World 는 이를 `0.001` 로 덮어씁니다(`diden_configs/sim_profiles/mujoco_warp/diden_spider.py`). "sim 끼리도 결과가 다르다" 는 말의 첫 번째 원인이 여기입니다.


## Part 2 — 상태 읽고 쓰기: `qpos`, `qvel`, 좌표계

강화학습 환경의 관측(observation)은 결국 `MjData` 에서 숫자를 꺼내 가공한 것입니다. World 에서 이 일을 맡는 곳이 `SimAdapter` (`diden_common/sim_adapter.py`) 이고, 좌표 변환은 `diden_common/math.py` 에 모여 있습니다. 여기서는 그 밑바닥을 numpy 로 만듭니다.

### 이름 → 주소

`qpos` 와 `qvel` 은 모든 관절의 좌표를 **트리 순서대로 이어 붙인** 벡터입니다. 관절 종류마다 차지하는 칸 수가 다릅니다.

| 관절 | `qpos` 칸 | `qvel` 칸 | 내용 |
|---|---|---|---|
| free | 7 | 6 | 위치 3 + 쿼터니언 4 (w, x, y, z) / 선속도 3 + 각속도 3 |
| ball | 4 | 3 | 쿼터니언 / 각속도 |
| hinge | 1 | 1 | 각도(rad) / 각속도 |
| slide | 1 | 1 | 변위(m) / 속도 |

관절 `j` 의 시작 주소는 `m.jnt_qposadr[j]`, `m.jnt_dofadr[j]` 에 있습니다. 이름으로는 `m.joint("이름").qposadr[0]` 처럼 얻습니다.

### 문제 2-1. 관절의 `qpos` / `qvel` 구간 찾기


In [ ]:
def joint_slices(m, joint_name: str):
    """관절 이름을 받아 (qpos 구간 slice, qvel 구간 slice) 를 돌려준다."""
    j = m.joint(joint_name)
    nq, nv = {
        mujoco.mjtJoint.mjJNT_FREE: (7, 6),
        mujoco.mjtJoint.mjJNT_BALL: (4, 3),
        mujoco.mjtJoint.mjJNT_HINGE: (1, 1),
        mujoco.mjtJoint.mjJNT_SLIDE: (1, 1),
    }[mujoco.mjtJoint(j.type[0])]
    qa, va = int(j.qposadr[0]), int(j.dofadr[0])
    return slice(qa, qa + nq), slice(va, va + nv)


In [ ]:
_xml = """
<mujoco>
  <worldbody>
    <body name="a"><joint name="hinge_a" type="hinge" axis="0 0 1"/><geom size="0.1"/>
      <body name="b" pos="0.3 0 0"><joint name="ball_b" type="ball"/><geom size="0.1"/>
        <body name="c" pos="0.3 0 0"><joint name="slide_c" type="slide" axis="1 0 0"/><geom size="0.1"/></body>
      </body>
    </body>
    <body name="f" pos="0 0 1"><freejoint name="free_f"/><geom size="0.1"/></body>
  </worldbody>
</mujoco>"""
_m = mujoco.MjModel.from_xml_string(_xml)
check("hinge_a", joint_slices(_m, "hinge_a") == (slice(0, 1), slice(0, 1)))
check("ball_b", joint_slices(_m, "ball_b") == (slice(1, 5), slice(1, 4)))
check("slide_c", joint_slices(_m, "slide_c") == (slice(5, 6), slice(4, 5)))
check("free_f", joint_slices(_m, "free_f") == (slice(6, 13), slice(5, 11)))
check("전체 길이", _m.nq == 13 and _m.nv == 11)


### `mj_forward`: 상태에서 나머지를 계산하기

`qpos` 는 **관절 좌표**입니다. 각 body 가 세계 좌표계 어디에 있는지(`d.xpos`, `d.xquat`, `d.xmat`)는 `qpos` 로부터 정기구학을 풀어야 나옵니다. `mj_step` 은 그 계산을 포함하지만, `qpos` 를 **직접 고친 뒤**에는 `mj_forward(m, d)` 를 불러야 `xpos` 등이 갱신됩니다.

리셋에서 이걸 빼먹으면 "리셋 직후 첫 관측만 이전 에피소드 값" 인 버그가 생깁니다. World 의 `ManagerBasedEnv.reset()` 이 `_reset_idx()` 다음에 `post_reset_sim_sync()` 를 부르는 이유입니다 (`diden_env/env/manager_based_env.py`).

### 문제 2-2. 자유 물체를 원하는 자세로 옮기기


In [ ]:
def set_free_body(m, d, joint_name: str, pos, quat=(1.0, 0.0, 0.0, 0.0), linvel=(0, 0, 0), angvel=(0, 0, 0)):
    """freejoint 의 qpos/qvel 을 써넣고, 파생량(xpos 등)까지 갱신한다. quat 은 (w, x, y, z)."""
    qs, vs = joint_slices(m, joint_name)
    d.qpos[qs] = np.concatenate([pos, quat])
    d.qvel[vs] = np.concatenate([linvel, angvel])
    mujoco.mj_forward(m, d)


In [ ]:
m = mujoco.MjModel.from_xml_string(drop_xml(1.0))
d = mujoco.MjData(m)
mujoco.mj_forward(m, d)
set_free_body(m, d, "ball_free", pos=[0.5, -0.2, 0.3])
check_close("qpos 가 바뀜", d.qpos, [0.5, -0.2, 0.3, 1, 0, 0, 0])
check_close("xpos 도 갱신됨 (mj_forward)", d.body("ball").xpos, [0.5, -0.2, 0.3])
check("시간은 흐르지 않음", d.time == 0.0)


### 쿼터니언과 좌표계

MuJoCo 의 쿼터니언은 `(w, x, y, z)` 순서이고, body 좌표계의 벡터를 세계 좌표계로 돌리는 회전입니다.

$$v^{W} = q \otimes v^{B} \otimes q^{-1}$$

벡터 부분을 $\mathbf{u} = (x, y, z)$ 라 하면 다음 식으로 곱셈 없이 계산할 수 있습니다.

$$v' = v + 2w\,(\mathbf{u} \times v) + 2\,\mathbf{u} \times (\mathbf{u} \times v)$$

World 에서는 이 함수가 `diden_common.math.quat_apply`, 반대 방향(세계 → body)이 `quat_apply_inverse` 입니다. 관측의 "몸통 기준 속도" 나 "투영 중력" 이 전부 `quat_apply_inverse` 한 줄에서 나옵니다.

### 문제 2-3. 쿼터니언으로 벡터 돌리기

배치 입력(`(..., 4)` 와 `(..., 3)`)에서도 동작하게 씁니다. 섹터 1 에서 같은 함수를 torch 로 다시 만듭니다.

<details><summary>힌트</summary>

`np.cross(a, b)` 는 마지막 축 기준으로 배치 연산이 됩니다. `w = q[..., :1]` 처럼 잘라 두면 브로드캐스팅이 맞습니다. 역회전은 켤레 쿼터니언 `(w, -x, -y, -z)` 로 돌리는 것과 같습니다.
</details>


In [ ]:
def quat_rotate(q, v):
    """body → world.  q: (..., 4) (w,x,y,z) 단위 쿼터니언,  v: (..., 3)."""
    q, v = np.asarray(q, dtype=float), np.asarray(v, dtype=float)
    w, u = q[..., :1], q[..., 1:]
    t = np.cross(u, v)
    return v + 2.0 * w * t + 2.0 * np.cross(u, t)


def quat_rotate_inverse(q, v):
    """world → body."""
    q = np.asarray(q, dtype=float)
    return quat_rotate(q * np.array([1.0, -1.0, -1.0, -1.0]), v)


In [ ]:
_rng = np.random.default_rng(0)
_q = _rng.normal(size=(5, 4)); _q /= np.linalg.norm(_q, axis=-1, keepdims=True)
_v = _rng.normal(size=(5, 3))
_want = np.zeros((5, 3))
for _i in range(5):
    mujoco.mju_rotVecQuat(_want[_i], _v[_i], _q[_i])
check_close("quat_rotate == mju_rotVecQuat", quat_rotate(_q, _v), _want, atol=1e-12)
check_close("역회전하면 원래대로", quat_rotate_inverse(_q, quat_rotate(_q, _v)), _v, atol=1e-12)
check_close("z축 90° 회전: x → y", quat_rotate([np.cos(np.pi / 4), 0, 0, np.sin(np.pi / 4)], [1, 0, 0]), [0, 1, 0], atol=1e-12)


### freejoint 의 `qvel` 은 좌표계가 섞여 있다

자주 틀리는 부분입니다. freejoint 의 `qvel` 6칸은

- 앞 3칸: 선속도, **세계 좌표계**
- 뒤 3칸: 각속도, **body 좌표계**

입니다. 구르는 공처럼 body 좌표계가 계속 도는 물체에서 각속도를 세계 좌표계로 쓰려면 한 번 돌려 줘야 합니다.

### 먼저 예상해 보기

공을 z 축으로 90° 돌려 놓고(`quat = (cos45°, 0, 0, sin45°)`) `qvel` 의 각속도 칸에 `(1, 0, 0)` 을 넣으면, 세계 좌표계에서 공은 어느 축을 중심으로 돌까요?

### 문제 2-4. 자유 물체의 세계 좌표계 속도


In [ ]:
def free_body_velocity_world(m, d, joint_name: str):
    """freejoint 가 달린 body 의 (선속도, 각속도) 를 둘 다 세계 좌표계로 돌려준다."""
    qs, vs = joint_slices(m, joint_name)
    quat = d.qpos[qs][3:7]
    linvel_w = d.qvel[vs][:3].copy()
    angvel_w = quat_rotate(quat, d.qvel[vs][3:6])
    return linvel_w, angvel_w


In [ ]:
m = mujoco.MjModel.from_xml_string(drop_xml(1.0))
d = mujoco.MjData(m)
set_free_body(m, d, "ball_free", pos=[0, 0, 1], quat=[np.cos(np.pi / 4), 0, 0, np.sin(np.pi / 4)],
              linvel=[0.3, -0.1, 0.2], angvel=[1.0, 0, 0])
_lin, _ang = free_body_velocity_world(m, d, "ball_free")
_ref = np.zeros(6)   # mj_objectVelocity 는 [각속도 3, 선속도 3] 순서로 돌려준다
mujoco.mj_objectVelocity(m, d, mujoco.mjtObj.mjOBJ_BODY, m.body("ball").id, _ref, 0)
check_close("각속도 (세계 좌표계) — 예상과 같았나요?", _ang, [0, 1, 0], atol=1e-12)
check_close("mj_objectVelocity 와 일치: 각속도", _ang, _ref[:3], atol=1e-12)
check_close("mj_objectVelocity 와 일치: 선속도", _lin, _ref[3:], atol=1e-12)


**정리**

- `qpos` / `qvel` 은 관절 좌표, `xpos` / `xquat` 은 세계 좌표계의 body 자세입니다. 앞의 것을 고치면 `mj_forward` 로 뒤의 것을 다시 계산해야 합니다.
- 쿼터니언은 `(w, x, y, z)`, body → world 방향입니다. Isaac 계열 코드나 `scipy` 는 `(x, y, z, w)` 를 쓰는 경우가 있어서, 엔진을 바꿀 때 가장 먼저 확인하는 항목입니다.
- freejoint 의 각속도는 body 좌표계입니다.


## Part 3 — geom: 질량, 보이는 것과 부딪히는 것, default class

### 질량은 어디서 오는가

body 의 질량·관성은 두 가지 방법으로 정해집니다.

1. body 안에 `<inertial pos=... mass=... fullinertia=.../>` 를 직접 쓴다. → Spider 의 `robot.xml` 이 이 방식입니다(CAD/URDF 에서 나온 값).
2. 쓰지 않으면 body 에 붙은 geom 들의 **밀도 × 부피** 로 컴파일러가 계산한다. 기본 밀도는 1000 kg/m³ (물).

`DIDEN_Asset/object/` 의 오브젝트들은 2번 방식입니다.

### 문제 3-1. 원하는 질량이 되는 밀도

반지름 $r$ 인 꽉 찬 구의 질량이 `mass` 가 되려면 밀도는 얼마여야 할까요? 그리고 그 구의 관성모멘트는 얼마일까요?


In [ ]:
def sphere_density(mass: float, radius: float) -> float:
    """질량이 mass 가 되도록 하는 균일 구의 밀도 [kg/m³]."""
    return mass / (4.0 / 3.0 * np.pi * radius**3)


def sphere_inertia(mass: float, radius: float) -> float:
    """균일한 구의 중심축 관성모멘트 [kg·m²]."""
    return 0.4 * mass * radius**2


In [ ]:
_m = mujoco.MjModel.from_xml_string(f"""
<mujoco><worldbody><body name="b"><freejoint/>
  <geom type="sphere" size="0.2" density="{sphere_density(1.0, 0.2)}"/>
</body></worldbody></mujoco>""")
check_close("질량 1 kg", _m.body("b").mass, [1.0], atol=1e-9)
check_close("관성 = 2/5 m r²", _m.body("b").inertia, [sphere_inertia(1.0, 0.2)] * 3, atol=1e-9)
print("   관성:", sphere_inertia(1.0, 0.2), "kg·m²")


### visual geom 과 collision geom

로봇의 겉모습은 수만 개 삼각형으로 된 메시(STL)이지만, 그걸로 충돌을 계산하면 느리고 불안정합니다. 그래서 같은 body 에 geom 을 두 벌 붙입니다.

- **visual**: 메시. 그리기만 하고, 부딪히지 않고, 질량에도 기여하지 않는다.
- **collision**: 상자·원기둥·구 같은 단순 도형. 부딪히기만 한다(보통 반투명하게 그려 디버깅에 쓴다).

이 구분은 geom 속성 세 가지로 만듭니다.

| 속성 | 뜻 |
|---|---|
| `contype`, `conaffinity` | 충돌 필터용 비트마스크 (Part 4). 둘 다 0 이면 아무것과도 부딪히지 않는다 |
| `density` | 0 이면 질량에 기여하지 않는다 |
| `group` | 렌더링에서 켜고 끄는 묶음 번호 (0~5). 물리와는 무관 |

### default class 와 include

geom 마다 이 속성을 반복해 쓰면 틀리기 쉽습니다. MJCF 는 `<default>` 로 기본값을 정하고, `class` 로 이름 붙인 묶음을 골라 쓰게 해 줍니다. 그리고 `<include file="..."/>` 로 여러 파일이 같은 기본값을 공유합니다. 실제 파일을 봅니다.


In [ ]:
print((ASSET / "object" / "common.xml").read_text())
print("─" * 100)
print((ASSET / "object" / "sphere.xml").read_text())


읽는 법:

- `<default>` 바로 아래의 `<geom .../>` 은 **이 파일을 include 한 모델의 모든 geom** 에 적용되는 기본값입니다.
- `<default class="visual">` 은 그 기본값을 물려받되 일부만 덮어쓴 묶음입니다. geom 에 `class="visual"` 이라고 쓰면 적용됩니다.
- 우선순위는 **geom 에 직접 쓴 값 > class 의 값 > 상위 default 의 값 > MuJoCo 내장 기본값** 입니다.

### 먼저 예상해 보기

`sphere.xml` 을 그대로 불러 컴파일하면

1. `sphere_visual` 과 `sphere_collision` 의 `contype`, `conaffinity`, `group` 은 각각 얼마일까요?
2. body `sphere` 의 질량은 얼마일까요? (반지름 0.2 m)

적어 본 뒤 실행합니다.


In [ ]:
m_obj = mujoco.MjModel.from_xml_path(str(ASSET / "object" / "sphere.xml"))
for name in ("sphere_visual", "sphere_collision"):
    g = m_obj.geom(name)
    print(f"{name:18s} contype={g.contype[0]} conaffinity={g.conaffinity[0]} group={g.group[0]} friction={g.friction}")
print("body mass =", m_obj.body("sphere").mass[0], "kg   nq =", m_obj.nq)


질량이 0.3 g 밖에 안 됩니다(`density="0.01"`). 그리고 `nq = 0` — 관절이 없어서 세계에 용접된 **움직이지 않는 물체** 입니다. 고정된 물체에는 질량이 의미가 없습니다. Core 는 이 오브젝트들을 씬에 붙여 로봇이 피해 가야 할 구조물로 씁니다(RRT 의 충돌 검사 대상). 섹터 1 에서 이 구를 **굴러다니는 공** 으로 쓰려면 freejoint 를 달고 밀도를 다시 정해야 합니다. 그 작업은 Part 8 에서 합니다.

### 문제 3-2. default class 로 공 만들기

`common.xml` 과 같은 방식으로, 아래 조건을 만족하는 MJCF 를 **default class 를 써서** 작성합니다.

| | `visual` class | `collision` class |
|---|---|---|
| contype / conaffinity | 0 / 0 | 1 / 1 |
| density | 0 | (함수 인자 `mass` 가 되도록) |
| group | 2 | 3 |
| rgba | 원하는 색 | `1 1 0 0.2` |

- body 이름 `ball`, freejoint `ball_free`, 위치 `(0, 0, radius)`
- geom 이름 `ball_visual`, `ball_collision` — 둘 다 같은 반지름의 구이고, 속성은 `class` 로만 지정합니다(geom 태그에는 `name`, `type`, `size`, `class` 만).
- 바닥 `floor` (plane) 도 하나 둡니다. 바닥은 `contype="1" conaffinity="1"` 로 직접 씁니다.

Spider 의 `DIDEN_Asset/robot/common/geom.xml` 이 정확히 이 구조입니다(`visual` 은 group 2, `collision` 은 group 3).


In [ ]:
def ball_with_classes_xml(mass: float = 1.0, radius: float = 0.2) -> str:
    return f"""
<mujoco model="ball_with_classes">
  <default>
    <default class="visual">
      <geom contype="0" conaffinity="0" density="0" group="2" rgba="0.8 0.3 0.2 1"/>
    </default>
    <default class="collision">
      <geom contype="1" conaffinity="1" density="{sphere_density(mass, radius)}" group="3" rgba="1 1 0 0.2"/>
    </default>
  </default>
  <worldbody>
    <light pos="0 0 3"/>
    <geom name="floor" type="plane" size="5 5 0.1" contype="1" conaffinity="1"/>
    <body name="ball" pos="0 0 {radius}">
      <freejoint name="ball_free"/>
      <geom name="ball_visual" type="sphere" size="{radius}" class="visual"/>
      <geom name="ball_collision" type="sphere" size="{radius}" class="collision"/>
    </body>
  </worldbody>
</mujoco>"""


In [ ]:
_xml = ball_with_classes_xml(mass=2.0, radius=0.15)
_m = mujoco.MjModel.from_xml_string(_xml)
_v, _c = _m.geom("ball_visual"), _m.geom("ball_collision")
check("default class 를 사용", 'class="visual"' in _xml and 'class="collision"' in _xml and "<default" in _xml)
check("visual: 충돌 안 함, group 2", _v.contype[0] == 0 and _v.conaffinity[0] == 0 and _v.group[0] == 2)
check("collision: 충돌 함, group 3", _c.contype[0] == 1 and _c.conaffinity[0] == 1 and _c.group[0] == 3)
check_close("질량은 collision geom 에서만", _m.body("ball").mass, [2.0], atol=1e-9)
check_close("관성", _m.body("ball").inertia, [sphere_inertia(2.0, 0.15)] * 3, atol=1e-9)


In [ ]:
# group 으로 골라 그리기: 왼쪽 = visual(group 2) 만, 오른쪽 = collision(group 3) 만
m = mujoco.MjModel.from_xml_string(ball_with_classes_xml())
d = mujoco.MjData(m)
mujoco.mj_forward(m, d)
cam = make_camera(distance=1.5)
media.show_images([render(m, d, cam, groups=[0, 2]), render(m, d, cam, groups=[0, 3])], titles=["visual (group 2)", "collision (group 3)"])


**정리**

- 질량은 `<inertial>` 또는 geom 밀도에서 옵니다. visual geom 의 `density="0"` 을 빼먹으면 질량이 두 배가 됩니다.
- `class` 는 속성 묶음의 이름입니다. `DIDEN_Asset/robot/common/` 의 네 파일(`material.xml`, `geom.xml`, `joint.xml`, `actuator.xml`)은 전부 class 정의만 담고 있고, 모든 로봇이 include 해서 씁니다. 모터 한 종류의 스펙(로터 관성, 토크 한계)을 한 군데서 고치기 위한 구조입니다.
- include 된 파일 안의 상대 경로(예: `../texture/steel.png`)는 **그 파일이 있는 폴더 기준** 으로 풀립니다.


## Part 4 — 접촉: 누가 누구와 부딪히는가, 마찰, 접촉력

### 충돌 필터: `contype` 과 `conaffinity`

두 geom A, B 는 다음 조건일 때만 충돌 검사를 합니다.

```
(A.contype & B.conaffinity) != 0   또는   (B.contype & A.conaffinity) != 0
```

`&` 는 비트 AND 입니다. `contype` 은 "나는 이런 종류다", `conaffinity` 는 "나는 이런 종류와 부딪힌다" 라고 읽으면 됩니다. 여기에 더해 MuJoCo 는 다음 쌍을 항상 제외합니다.

- 같은 body 에 붙은 geom 끼리
- 부모-자식 body 사이 (기본 설정)
- 둘 다 세계에 고정된 geom 끼리

### 문제 4-1. 충돌 필터 규칙


In [ ]:
def can_collide(contype_a: int, conaffinity_a: int, contype_b: int, conaffinity_b: int) -> bool:
    """비트마스크 규칙만으로 두 geom 이 충돌 후보인지 판정한다."""
    return bool((contype_a & conaffinity_b) or (contype_b & conaffinity_a))


In [ ]:
check("기본값 (1,1) 끼리는 충돌", can_collide(1, 1, 1, 1))
check("visual (0,0) 은 아무것과도 안 함", not can_collide(0, 0, 1, 1))
check("object(1,0) 끼리는 안 함", not can_collide(1, 0, 1, 0))
check("object(1,0) 와 바닥(1,1) 은 충돌", can_collide(1, 0, 1, 1))
check("한쪽만 맞아도 충돌 (0,3) vs (2,0)", can_collide(0, 3, 2, 0))
check("비트가 안 겹치면 안 함 (2,2) vs (1,1)", not can_collide(2, 2, 1, 1))


### 먼저 예상해 보기

`DIDEN_Asset/object/` 의 오브젝트는 전부 `contype=1, conaffinity=0` 입니다. Spider 로봇의 geom 기본값은 `contype=1, conaffinity=1` 입니다(`robot.xml` 의 `<default>`).

1. 오브젝트 위에 다른 오브젝트를 떨어뜨리면 어떻게 될까요?
2. 오브젝트 위에 로봇이 올라설 수 있을까요?
3. 왜 이렇게 설계했을까요?

아래는 Asset 의 `box.xml` 위로 Asset 의 `sphere.xml` 을 떨어뜨리는 실험입니다. `MjSpec` 으로 파일 두 개를 한 씬에 붙이는 방법은 Part 8 에서 직접 만듭니다. 지금은 결과만 봅니다.


In [ ]:
def _two_objects_scene(ball_conaffinity: int):
    spec = mujoco.MjSpec()
    spec.worldbody.add_light(pos=[0, 0, 4])
    spec.worldbody.add_geom(name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE, size=[5, 5, 0.1])
    box = mujoco.MjSpec.from_file(str(ASSET / "object" / "box.xml"))
    spec.worldbody.add_frame().attach_body(box.body("box"), "box/", "")
    sph = mujoco.MjSpec.from_file(str(ASSET / "object" / "sphere.xml"))
    ball = spec.worldbody.add_frame(pos=[0.3, 0.2, 1.0]).attach_body(sph.body("sphere"), "ball/", "")
    ball.add_freejoint(name="ball_free")
    for g in ball.geoms:
        if g.contype:                      # collision geom 만
            g.conaffinity = ball_conaffinity
            g.density = sphere_density(1.0, 0.2)
    return spec.compile()

for _ca in (0, 1):
    m = _two_objects_scene(ball_conaffinity=_ca)
    d = mujoco.MjData(m)
    for _ in range(1500):
        mujoco.mj_step(m, d)
    print(f"공의 conaffinity={_ca}:  3초 뒤 공 높이 = {d.body('ball/sphere').xpos[2]:.3f} m   (상자 윗면 0.3 + 반지름 0.2 = 0.5, 바닥 + 반지름 = 0.2)")


`conaffinity=0` 이면 공이 상자를 **통과** 해서 바닥에 놓입니다. 오브젝트끼리는 서로 검사하지 않고, `conaffinity` 에 1번 비트가 있는 geom(로봇)만 오브젝트와 부딪힙니다. 고정된 구조물끼리는 어차피 MuJoCo 가 검사하지 않으므로, 이 설정이 차이를 만드는 것은 지금처럼 오브젝트를 움직이는 물체로 쓸 때입니다. 3번 질문(왜 이렇게 정했는가)은 코드만으로는 답이 나오지 않습니다. Asset 담당자에게 물어볼 질문으로 `QUESTION_TREE.md` 에 적어 둡니다. 섹터 1 의 공은 바닥 위를 굴러야 하므로 바닥의 `conaffinity` 가 1 이면 충분합니다.

### 접촉력 읽기

스텝을 밟으면 `d.contact[:d.ncon]` 에 현재 접촉점들이 들어 있습니다. 각 접촉은 `geom1`, `geom2`(geom id), `pos`, `frame`(접촉 좌표계, 첫 축이 법선), `dist`(음수면 파고든 깊이)를 가집니다. 힘은 `mj_contactForce(m, d, i, out6)` 로 꺼내며, `out6[0]` 이 법선 방향 힘, `out6[1:3]` 이 접선(마찰) 힘입니다.

World 에서 "발이 땅에 닿았는가" 를 판단하는 접촉 센서, 보상의 접촉력 항이 전부 여기서 나옵니다(`diden_common/sim_adapter.py` 의 `get_contact_info`).

### 문제 4-2. 접촉 목록 만들기


In [ ]:
def contact_list(m, d):
    """현재 접촉을 [(geom1 이름, geom2 이름, 법선력[N]), ...] 로 돌려준다."""
    out = []
    f6 = np.zeros(6)
    for i in range(d.ncon):
        c = d.contact[i]
        mujoco.mj_contactForce(m, d, i, f6)
        out.append((m.geom(c.geom1).name, m.geom(c.geom2).name, float(f6[0])))
    return out


In [ ]:
m = mujoco.MjModel.from_xml_string(ball_with_classes_xml(mass=1.0, radius=0.2))
d = mujoco.MjData(m)
for _ in range(1000):
    mujoco.mj_step(m, d)
_c = contact_list(m, d)
print("  ", _c)
check("접촉 1개: floor 와 ball_collision", len(_c) == 1 and set(_c[0][:2]) == {"floor", "ball_collision"})
check_close("정지한 1 kg 공의 법선력 = m·g", _c[0][2], 9.81, atol=1e-3)


### 마찰

geom 의 `friction="a b c"` 는 세 개의 계수입니다.

| 칸 | 이름 | 막는 운동 | 쓰이려면 |
|---|---|---|---|
| 1 | sliding | 접선 방향 미끄러짐 | `condim >= 3` (기본) |
| 2 | torsional | 법선축 둘레 비틀림 | `condim >= 4` |
| 3 | rolling | 구름 | `condim = 6` |

두 geom 이 만났을 때 쓰는 계수는 **둘 중 큰 값** 입니다(평균이 아닙니다). 단 `priority` 가 높은 geom 이 있으면 그쪽 값만 씁니다. World 의 Spider 설정은 로봇의 collision geom 에 `priority=1` 을 줍니다(`diden_mjlab/robots/diden_spider/cfg.py` 의 `CUSTOM_COLLISION`). 그러면 지형의 마찰값이 무엇이든 로봇 쪽 값이 쓰이고, 마찰 랜덤화도 로봇 쪽 값 하나만 바꾸면 됩니다.

Spider 의 `robot.xml` 은 `condim="4"` 로 torsional 마찰까지 켭니다. 발 하나로 붙어 있을 때 그 발이 법선축 둘레로 돌아가는 것을 막는 항입니다.

### 경사면 실험

바닥을 기울이는 대신 **중력을 기울입니다**. 경사각 $\theta$ 는 `m.opt.gravity = g·(sinθ, 0, -cosθ)` 와 같습니다.


In [ ]:
def tilt_gravity(m, theta_rad: float, g: float = 9.81):
    """x 방향으로 theta 만큼 기운 경사면 위에 있는 것과 같게 중력을 돌린다."""
    m.opt.gravity[:] = g * np.array([np.sin(theta_rad), 0.0, -np.cos(theta_rad)])


def box_on_floor_xml(mu_box: float, mu_floor: float) -> str:
    return f"""
<mujoco><option timestep="0.002"/><worldbody>
  <light pos="0 0 3"/>
  <geom name="floor" type="plane" size="50 5 0.1" friction="{mu_floor}"/>
  <body name="box" pos="0 0 0.05"><freejoint name="box_free"/>
    <geom name="box" type="box" size="0.05 0.05 0.05" friction="{mu_box}"/></body>
</worldbody></mujoco>"""


### 문제 4-3. 미끄러질까?

경사각과 두 geom 의 마찰계수를 받아, 상자가 1초 동안 미끄러지는 거리를 **해석적으로** 예측합니다. 미끄러지지 않으면 0 을 돌려줍니다.

<details><summary>힌트</summary>

미끄러지는 조건은 $\tan\theta > \mu$, 미끄러질 때 가속도는 $a = g(\sin\theta - \mu\cos\theta)$, 거리는 $\tfrac12 a t^2$ 입니다. $\mu$ 는 두 계수 중 **큰 값**.
</details>


In [ ]:
def predict_box_slide(theta_rad: float, mu_box: float, mu_floor: float, t: float = 1.0, g: float = 9.81) -> float:
    mu = max(mu_box, mu_floor)
    if np.tan(theta_rad) <= mu:
        return 0.0
    return 0.5 * g * (np.sin(theta_rad) - mu * np.cos(theta_rad)) * t**2


In [ ]:
for _mu_b, _mu_f, _deg in [(0.5, 0.2, 20), (0.5, 0.2, 30), (0.2, 0.5, 40), (0.3, 0.3, 15), (0.3, 0.3, 25)]:
    m = mujoco.MjModel.from_xml_string(box_on_floor_xml(_mu_b, _mu_f))
    d = mujoco.MjData(m)
    tilt_gravity(m, np.radians(_deg))
    for _ in range(500):
        mujoco.mj_step(m, d)
    _pred = predict_box_slide(np.radians(_deg), _mu_b, _mu_f)
    check_close(f"μ_box={_mu_b} μ_floor={_mu_f} θ={_deg}°: sim {d.qpos[0]:.4f} m / 예측 {_pred:.4f} m", d.qpos[0], _pred, atol=0.01)


"미끄러지지 않는다" 고 예측한 경우에도 시뮬레이션에서는 1~5 mm 움직입니다. 버그가 아니라 MuJoCo 의 접촉이 **부드러운 제약** 이기 때문입니다. 마찰 제약도 약간의 속도를 허용해서, 정지 마찰 상태에서도 아주 천천히 흐릅니다(creep). 벽에 붙어 있어야 하는 로봇에게는 이 흐름이 문제가 되므로 두 레포가 각자 대책을 씁니다.

- Core / `robot.xml`: `noslip_iterations="20"` (접촉 풀이 뒤에 미끄럼만 다시 잡는 후처리)
- World: `cone="elliptic"`, `impratio=10` (마찰 제약을 법선 제약보다 10배 단단하게) — `diden_configs/sim_profiles/mujoco_warp/diden_spider.py`

### 구르기

이제 공입니다. 경사각 $\theta$ 에서 반지름 $r$, 질량 $m$, 관성 $I = \tfrac25 m r^2$ 인 공이 **미끄러지지 않고** 구르면

$$m a = m g\sin\theta - f, \qquad I\,\frac{a}{r} = f\,r \quad\Rightarrow\quad a = \frac{g\sin\theta}{1 + I/(m r^2)} = \tfrac57\, g\sin\theta$$

이때 필요한 마찰력은 $f = \tfrac27 m g\sin\theta$ 이고, 이것이 $\mu m g\cos\theta$ 를 넘으면 미끄러지면서 구릅니다.

### 문제 4-4. 구르는 공의 가속도


In [ ]:
def predict_ball_accel(theta_rad: float, mu: float, g: float = 9.81) -> float:
    """경사각 theta, 마찰계수 mu 일 때 균일한 구의 중심 가속도. 미끄러지는 경우까지 처리한다."""
    if mu >= 2.0 / 7.0 * np.tan(theta_rad):
        return 5.0 / 7.0 * g * np.sin(theta_rad)
    return g * (np.sin(theta_rad) - mu * np.cos(theta_rad))


In [ ]:
def _ball_accel_sim(theta, mu):
    m = mujoco.MjModel.from_xml_string(f"""
<mujoco><option timestep="0.002"/><worldbody>
  <geom name="floor" type="plane" size="50 5 0.1" friction="{mu}"/>
  <body name="ball" pos="0 0 0.1"><freejoint/><geom type="sphere" size="0.1" friction="{mu}"/></body>
</worldbody></mujoco>""")
    d = mujoco.MjData(m)
    tilt_gravity(m, theta)
    for _ in range(100):
        mujoco.mj_step(m, d)
    v0 = d.qvel[0]
    for _ in range(500):
        mujoco.mj_step(m, d)
    return (d.qvel[0] - v0) / (500 * m.opt.timestep), d.qvel[0] - d.qvel[4] * 0.1   # 가속도, 접촉점 미끄럼 속도

for _mu, _deg in [(1.0, 10), (0.5, 30), (0.2, 30), (0.1, 30), (0.05, 20)]:
    _a, _slip = _ball_accel_sim(np.radians(_deg), _mu)
    _pred = predict_ball_accel(np.radians(_deg), _mu)
    check_close(f"μ={_mu} θ={_deg}°: sim a={_a:.3f} / 예측 {_pred:.3f}  (접촉점 미끄럼 속도 {_slip:+.3f} m/s)", _a, _pred, rtol=0.01)


### 토크로 공 굴리기 — 섹터 1 의 액션

섹터 1 의 정책은 공에 **토크** 를 걸어 굴립니다. 바닥이 수평일 때 공에 y 축 토크 $\tau$ 를 걸면, 접촉점의 마찰력 $f$ 가 공을 +x 로 밉니다.

$$m a = f, \qquad I\,\frac{a}{r} = \tau - f r \quad\Rightarrow\quad a = \frac{\tau}{r\,(m + I/r^2)} = \frac{\tau}{\tfrac75\, m r}$$

필요한 마찰력 $f = m a$ 가 $\mu m g$ 를 넘으면 헛돕니다.

`MjData` 에는 body 에 외력을 직접 거는 칸이 있습니다: `d.xfrc_applied[body_id] = [Fx, Fy, Fz, Tx, Ty, Tz]`. **세계 좌표계** 기준이고 body 의 질량중심에 작용합니다. 한 번 써넣으면 지울 때까지 유지됩니다.

### 문제 4-5. 헛돌지 않는 최대 토크


In [ ]:
def predict_ball_accel_from_torque(torque: float, mass: float, radius: float) -> float:
    """미끄러지지 않을 때, y 축 토크가 만드는 x 방향 가속도."""
    return torque / (1.4 * mass * radius)


def max_torque_without_slip(mu: float, mass: float, radius: float, g: float = 9.81) -> float:
    """이 값을 넘으면 공이 헛돈다."""
    return 1.4 * mu * mass * g * radius


In [ ]:
def _torque_roll(torque, mu=0.5, mass=1.0, radius=0.2, seconds=1.0):
    m = mujoco.MjModel.from_xml_string(f"""
<mujoco><option timestep="0.002"/><worldbody>
  <geom name="floor" type="plane" size="50 50 0.1" friction="{mu}"/>
  <body name="ball" pos="0 0 {radius}"><freejoint/>
    <geom type="sphere" size="{radius}" friction="{mu}" density="{sphere_density(mass, radius)}"/></body>
</worldbody></mujoco>""")
    d = mujoco.MjData(m)
    d.xfrc_applied[m.body("ball").id, 4] = torque          # 세계 y 축 토크
    for _ in range(round(seconds / m.opt.timestep)):
        mujoco.mj_step(m, d)
    lin, ang = d.qvel[:3], quat_rotate(d.qpos[3:7], d.qvel[3:6])
    return lin[0] / seconds, lin[0] - ang[1] * radius        # 평균 가속도, 접촉점 미끄럼 속도

_tmax = max_torque_without_slip(0.5, 1.0, 0.2)
check_close("τ_max (μ=0.5, 1 kg, r=0.2)", _tmax, 1.3734, atol=1e-4)
_a, _slip = _torque_roll(0.5 * _tmax)
check_close(f"τ = 0.5·τ_max: 가속도 {_a:.3f} m/s², 미끄럼 {_slip:+.4f} m/s", _a, predict_ball_accel_from_torque(0.5 * _tmax, 1.0, 0.2), rtol=0.01)
check("   → 거의 미끄러지지 않음", abs(_slip) < 0.01)
_a, _slip = _torque_roll(2.0 * _tmax)
check_close(f"τ = 2·τ_max: 가속도 {_a:.3f} m/s² 는 μ·g 에서 포화, 미끄럼 {_slip:+.3f} m/s", _a, 0.5 * 9.81, rtol=0.01)
check("   → 헛돈다", _slip < -1.0)


**정리**

- 충돌 여부는 `contype` / `conaffinity` 비트마스크로 정하고, 마찰계수는 두 geom 중 큰 값(또는 `priority` 높은 쪽)을 씁니다.
- 공을 토크로 굴릴 때 낼 수 있는 가속도는 $\mu g$ 를 넘지 못합니다. 섹터 1 에서 **액션 스케일** (정책 출력 1 이 몇 N·m 인가)을 정할 때 이 한계가 기준이 됩니다. 한계보다 훨씬 큰 토크를 허용하면 정책은 헛도는 구간에서 시간을 보내고, 마찰을 랜덤화하면 한계 자체가 환경마다 달라집니다.

**부숴 보기**

1. `_torque_roll` 의 MJCF 에서 바닥 마찰만 0.1 로 낮추면 결과가 달라질까요? (힌트: 큰 값 규칙)
2. `condim="6"` 과 rolling 마찰 `friction="0.5 0.005 0.01"` 을 주고 토크 없이 굴려 보면 공이 멈춥니다. 섹터 1 의 공은 `condim=3` 이라 한번 구르면 스스로 멈추지 않습니다. 정책이 **제동** 을 배워야 한다는 뜻입니다.


## Part 5 — 관절과 기구학 트리: Spider 다리 한 개

로봇은 body 가 부모-자식으로 이어진 **트리** 입니다. 자식 body 의 `pos` 는 부모 좌표계 기준이고, 자식 안에 쓴 `<joint>` 가 "부모에 대해 어떻게 움직이는가" 를 정합니다.

Spider 의 다리 하나는 모터 3개(엉덩이 yaw, 허벅지, 종아리)와 수동 발목 3개로 되어 있습니다. `robot.xml` 의 앞왼쪽(FL) 다리에서 모터 관절만 뽑으면 이렇습니다.

```
Torso
└─ Leg_FL_Link_01   pos = ( 0.10675, 0.10675,  0.0875)   joint Leg_FL_Joint_01  axis z   ← 엉덩이 yaw
   └─ Leg_FL_Link_02   pos = ( 0.0845, 0,  0.066)        joint Leg_FL_Joint_02  axis x   ← 허벅지
      └─ Leg_FL_Link_03   pos = (-0.0865, 0, -0.383)     joint Leg_FL_Joint_03  axis x   ← 종아리
         └─ site Foot_FL_Tip_site   pos = (0, 0, -0.4821)                                ← 발끝
```

### XML 대신 `MjSpec`

지금까지는 MJCF 를 문자열로 썼습니다. MuJoCo 3 부터는 같은 것을 파이썬 객체로 조립하는 `MjSpec` 이 있고, World 는 **씬을 전부 `MjSpec` 으로** 만듭니다(Part 8). 대응 관계는 단순합니다.

| MJCF | MjSpec |
|---|---|
| `<body name=.. pos=..>` (부모 안에) | `parent.add_body(name=.., pos=[..])` |
| `<joint name=.. type="hinge" axis=..>` | `body.add_joint(name=.., type=mujoco.mjtJoint.mjJNT_HINGE, axis=[..])` |
| `<geom type="capsule" fromto=.. size=..>` | `body.add_geom(type=mujoco.mjtGeom.mjGEOM_CAPSULE, fromto=[..], size=[r, 0, 0])` |
| `<site name=.. pos=..>` | `body.add_site(name=.., pos=[..])` |
| `<option timestep=..>` | `spec.option.timestep = ..` |
| `<compiler angle="radian">` | `spec.compiler.degree = False` |
| 컴파일 | `model = spec.compile()` |

**함정**: MJCF 와 `MjSpec` 모두 각도의 기본 단위가 **도(degree)** 입니다. `robot.xml` 이 첫머리에 `<compiler angle="radian">` 을 쓰는 이유입니다. `MjSpec` 에서 `spec.compiler.degree = False` 를 빼먹고 `range=[-6.28, 6.28]` 을 쓰면 관절이 ±6.28° 안에 갇힙니다.

### 문제 5-1. 다리 한 개 조립

`MjSpec` 으로 위 트리를 만듭니다.

- `spec.compiler.degree = False`, `timestep = 0.001`, 적분기 `implicitfast` (`mujoco.mjtIntegrator.mjINT_IMPLICITFAST`)
- `Torso`: 세계에 **고정** (관절 없음), 위치 `(0, 0, 1.0)`. 상자 geom 하나(크기 자유).
- 관절 3개: hinge, 위 표의 축, `range = [-6.28, 6.28]`, `armature = 0.235786` (`joint.xml` 의 `diden_actuator_v2`)
- 링크 geom: 캡슐로 대략 그리면 됩니다. 질량은 `mass=` 로 직접 줍니다 — Link_01: 2.246938, Link_02: 2.5932, Link_03: 0.452306 kg.
- 모든 geom 은 `contype=0, conaffinity=0` (이 Part 에서는 충돌을 쓰지 않습니다)
- site `Foot_FL_Tip_site`


In [ ]:
LEG_OFFSETS = {                      # robot.xml 의 Leg_FL_* 값
    "Leg_FL_Link_01": (0.10675, 0.10675, 0.0875),
    "Leg_FL_Link_02": (0.0845, 0.0, 0.066),
    "Leg_FL_Link_03": (-0.0865, 0.0, -0.383),
    "Foot_FL_Tip_site": (0.0, 0.0, -0.4821),
}
LEG_JOINT_AXES = {"Leg_FL_Joint_01": (0, 0, 1), "Leg_FL_Joint_02": (1, 0, 0), "Leg_FL_Joint_03": (1, 0, 0)}
LEG_LINK_MASSES = {"Leg_FL_Link_01": 2.246938, "Leg_FL_Link_02": 2.5932, "Leg_FL_Link_03": 0.452306}
MOTOR_ARMATURE = 0.235786


def build_leg_spec() -> mujoco.MjSpec:
    """Torso 가 공중에 고정된 FL 다리 한 개. (액추에이터는 Part 6 에서 붙인다.)"""
    spec = mujoco.MjSpec()
    spec.compiler.degree = False
    spec.option.timestep = 0.001
    spec.option.integrator = mujoco.mjtIntegrator.mjINT_IMPLICITFAST
    G, J = mujoco.mjtGeom, mujoco.mjtJoint
    nocol = dict(contype=0, conaffinity=0)

    spec.worldbody.add_light(pos=[0, 0, 4])
    torso = spec.worldbody.add_body(name="Torso", pos=[0, 0, 1.0])
    torso.add_geom(name="torso_geom", type=G.mjGEOM_BOX, size=[0.17, 0.17, 0.05], rgba=[0.2, 0.2, 0.2, 1], **nocol)

    parent, tip = torso, None
    names = ["Leg_FL_Link_01", "Leg_FL_Link_02", "Leg_FL_Link_03"]
    ends = [LEG_OFFSETS["Leg_FL_Link_02"], LEG_OFFSETS["Leg_FL_Link_03"], LEG_OFFSETS["Foot_FL_Tip_site"]]
    for i, (name, end) in enumerate(zip(names, ends), start=1):
        body = parent.add_body(name=name, pos=list(LEG_OFFSETS[name]))
        jname = f"Leg_FL_Joint_0{i}"
        body.add_joint(name=jname, type=J.mjJNT_HINGE, axis=list(LEG_JOINT_AXES[jname]),
                       range=[-6.28, 6.28], armature=MOTOR_ARMATURE)
        body.add_geom(name=f"{name}_geom", type=G.mjGEOM_CAPSULE, fromto=[0, 0, 0, *end], size=[0.02, 0, 0],
                      mass=LEG_LINK_MASSES[name], rgba=[0.8, 0.5, 0.2, 1], **nocol)
        parent = body
    parent.add_site(name="Foot_FL_Tip_site", pos=list(LEG_OFFSETS["Foot_FL_Tip_site"]), size=[0.03, 0, 0], rgba=[1, 0, 0, 1])
    return spec


In [ ]:
import warnings

m_leg = build_leg_spec().compile()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    m_ref = mujoco.MjModel.from_xml_path(str(ASSET / "robot" / "diden_spider_v2" / "robot.xml"))   # 진짜 로봇
check("관절 3개, 자유도 3", m_leg.njnt == 3 and m_leg.nq == 3 and m_leg.nv == 3)
check("Torso 는 세계에 고정", m_leg.body("Torso").jntnum[0] == 0)
for _n in ("Leg_FL_Link_01", "Leg_FL_Link_02", "Leg_FL_Link_03"):
    check_close(f"{_n} 위치가 robot.xml 과 같음", m_leg.body(_n).pos, m_ref.body(_n).pos)
    check_close(f"{_n} 질량", m_leg.body(_n).mass, m_ref.body(_n).mass, atol=1e-6)
for _n in LEG_JOINT_AXES:
    check_close(f"{_n} 축", m_leg.joint(_n).axis, m_ref.joint(_n).axis)
    check_close(f"{_n} 범위 (라디안!)", m_leg.joint(_n).range, m_ref.joint(_n).range)
check_close("armature", m_leg.dof_armature, [MOTOR_ARMATURE] * 3)
check_close("발끝 site", m_leg.site("Foot_FL_Tip_site").pos, m_ref.site("Foot_FL_Tip_site").pos)
check("부모 관계", m_leg.body("Leg_FL_Link_03").parentid[0] == m_leg.body("Leg_FL_Link_02").id)


In [ ]:
d_leg = mujoco.MjData(m_leg)
d_leg.qpos[:] = [0.6, -0.869, 0.994]
mujoco.mj_forward(m_leg, d_leg)
media.show_image(render(m_leg, d_leg, make_camera(lookat=(0.3, 0.2, 0.8), distance=2.2, azimuth=150)))


### 정기구학을 손으로

관절각 $q = (q_1, q_2, q_3)$ 가 주어졌을 때 발끝이 Torso 좌표계 어디에 있는지 구합니다. 트리를 따라 내려가며 "부모 좌표계에서 본 자식 원점 이동 → 관절 회전" 을 번갈아 곱하면 됩니다.

$$p_{\text{tip}} = p_1 + R_z(q_1)\Big(p_2 + R_x(q_2)\big(p_3 + R_x(q_3)\,p_{\text{tip}}^{(3)}\big)\Big)$$

Core 의 NMPC 와 RRT, World 의 `relative_foot_pos` 관측과 IK 리셋이 모두 이 식에서 출발합니다(`diden_common/robots/diden_spider/kinematics.py`).

### 먼저 예상해 보기

$q = (0, 0, 0)$ 일 때 발끝은 Torso 기준 어디에 있을까요? 표의 숫자만 더하면 됩니다.

### 문제 5-2. 다리 정기구학


In [ ]:
def rot_z(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])


def rot_x(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1.0, 0.0, 0.0], [0.0, c, -s], [0.0, s, c]])


def leg_fk(q) -> np.ndarray:
    """FL 다리 관절각 (3,) → Torso 좌표계에서 본 발끝 위치 (3,)."""
    p1, p2, p3, pt = (np.array(LEG_OFFSETS[k]) for k in ("Leg_FL_Link_01", "Leg_FL_Link_02", "Leg_FL_Link_03", "Foot_FL_Tip_site"))
    return p1 + rot_z(q[0]) @ (p2 + rot_x(q[1]) @ (p3 + rot_x(q[2]) @ pt))


In [ ]:
check_close("q=0: 오프셋의 합", leg_fk(np.zeros(3)), [0.10675 + 0.0845 - 0.0865, 0.10675, 0.0875 + 0.066 - 0.383 - 0.4821], atol=1e-12)
_rng = np.random.default_rng(1)
for _ in range(5):
    _q = _rng.uniform(-1.5, 1.5, 3)
    d_leg.qpos[:] = _q
    mujoco.mj_forward(m_leg, d_leg)
    _want = d_leg.site("Foot_FL_Tip_site").xpos - d_leg.body("Torso").xpos      # Torso 는 회전이 없으므로 빼기만 하면 된다
    check_close(f"q={_q.round(2)}: MuJoCo 와 일치", leg_fk(_q), _want, atol=1e-10)


### 야코비안

발끝 속도와 관절 속도의 관계 $\dot p = J(q)\,\dot q$ 의 $J$ 가 야코비안입니다. 두 레포에서 계속 나옵니다.

- 발에 걸리는 힘 $f$ 를 내려면 관절 토크 $\tau = J^\top f$ (Core NMPC 의 `τ_ff = −Jᵀf`)
- 관절 토크에서 접촉력을 거꾸로 추정 (Core 추정기의 접촉 판정)
- 역기구학의 반복 풀이 (Core 의 damped least squares)

### 문제 5-3. 수치 야코비안

`leg_fk` 를 중앙차분해서 3×3 야코비안을 만듭니다. MuJoCo 의 `mj_jacSite` 와 비교합니다.


In [ ]:
def leg_jacobian(q, eps: float = 1e-6) -> np.ndarray:
    """J[i, j] = ∂(발끝 위치 i) / ∂(q j).  중앙차분."""
    q = np.asarray(q, dtype=float)
    J = np.zeros((3, 3))
    for j in range(3):
        dq = np.zeros(3)
        dq[j] = eps
        J[:, j] = (leg_fk(q + dq) - leg_fk(q - dq)) / (2 * eps)
    return J


In [ ]:
_q = np.array([0.6, -0.869, 0.994])
d_leg.qpos[:] = _q
mujoco.mj_forward(m_leg, d_leg)
_jacp = np.zeros((3, m_leg.nv))
mujoco.mj_jacSite(m_leg, d_leg, _jacp, None, m_leg.site("Foot_FL_Tip_site").id)
check_close("mj_jacSite 와 일치", leg_jacobian(_q), _jacp, atol=1e-7)
_f = np.array([0.0, 0.0, 100.0])       # 발끝을 위로 100 N 미는 힘
print("   발끝을 위로 100 N 밀 때 필요한 관절 토크 τ = Jᵀ f =", leg_jacobian(_q).T @ _f, "N·m")


**정리**

- body 트리 + 관절 축 + 오프셋이 기구학의 전부입니다. `robot.xml` 을 읽을 때 `pos` 와 `axis` 만 따라가면 로봇의 형태가 나옵니다.
- 각도 단위(도/라디안)는 컴파일러 설정입니다.
- `armature` 는 감속기를 거쳐 관절축에서 본 **모터 회전자의 관성** 입니다. 링크 관성과 별개로 관절 공간 관성행렬의 대각에 더해집니다. 다음 Part 에서 이 값이 왜 중요한지 봅니다.


## Part 6 — 액추에이터와 PD 제어

관절에 힘을 넣는 방법이 액추에이터입니다. MuJoCo 의 액추에이터는 스칼라 입력 `ctrl` 을 받아 관절에 걸 힘을 만듭니다.

$$\text{force} = \underbrace{\text{gain}}_{\text{gainprm}} \cdot \text{ctrl} + \underbrace{b_0 + b_1\,q + b_2\,\dot q}_{\text{biasprm}}$$

자주 쓰는 두 종류는 이 식의 특수한 경우입니다.

| MJCF | gain | bias | 뜻 |
|---|---|---|---|
| `<motor>` | 1 | 0 | `ctrl` 이 곧 토크 |
| `<position kp=.. kv=..>` | kp | $-k_p q - k_v \dot q$ | `ctrl` 이 목표 각도. force = $k_p(\text{ctrl} - q) - k_v\dot q$ |

**같은 로봇인데 쓰는 곳마다 방식이 다릅니다.** 이 차이가 sim-to-real 의 핵심 중 하나입니다.

| 곳 | 방식 |
|---|---|
| `DIDEN_Asset` 의 `robot.xml` | `<motor class="diden_motor">` — 토크 입력, `ctrlrange="-386 386"` |
| World | 모터를 **지우고** MuJoCo 내장 position 액추에이터(kp=100, kd=20)를 넣는다 (`diden_mjlab/robots/diden_spider/spec.py` 의 `_strip_actuators`, `cfg.py` 의 `BuiltinPositionActuatorCfg`) |
| Core (시뮬레이터) | Simulator 스레드가 **직접** $\tau = k_p(q^* - q) + k_d(\dot q^* - \dot q) + \tau_{ff}$ 를 계산해 넣는다 (`src/engine/simulator/thread.cpp`) |
| 실제 로봇 | PC 는 `{pos, vel, kp, kd, tau}` 를 보내고 PD 는 **모터 드라이버 안에서** 돈다 (`driver/drec_baseboard/tx/protocol.h` 의 `MotorCommand`) |

### 문제 6-1. PD 토크


In [ ]:
def pd_torque(q, qd, q_des, kp, kd, torque_limit=None, qd_des=0.0, tau_ff=0.0):
    """τ = kp (q_des − q) + kd (qd_des − qd) + τ_ff,  그리고 ±torque_limit 로 자른다."""
    tau = kp * (np.asarray(q_des) - np.asarray(q)) + kd * (qd_des - np.asarray(qd)) + tau_ff
    if torque_limit is not None:
        tau = np.clip(tau, -torque_limit, torque_limit)
    return tau


In [ ]:
check_close("P 항", pd_torque([0.0], [0.0], [0.5], 100, 20), [50.0])
check_close("D 항", pd_torque([0.5], [2.0], [0.5], 100, 20), [-40.0])
check_close("포화", pd_torque([0.0, 0.0], [0.0, 0.0], [10.0, -10.0], 100, 20, torque_limit=386), [386.0, -386.0])
check_close("피드포워드", pd_torque([0.0], [0.0], [0.0], 100, 20, tau_ff=7.0), [7.0])


### 문제 6-2. 다리에 액추에이터 달기

`build_leg_spec()` 이 만든 spec 에 관절 3개의 액추에이터를 붙입니다. `kind` 에 따라 두 종류를 만듭니다.

- `"motor"`: `ctrl` = 토크. `ctrlrange = [-386, 386]`, `ctrllimited = True`
- `"position"`: `ctrl` = 목표 각도. `gainprm[0] = kp`, `biastype = AFFINE`, `biasprm[:3] = [0, -kp, -kd]`, `forcerange = [-386, 386]`, `forcelimited = True`

액추에이터 이름은 `robot.xml` 처럼 `<관절 이름>_Actuator` 로 합니다.

<details><summary>힌트</summary>

```python
a = spec.add_actuator(name=..., target=<관절 이름>, trntype=mujoco.mjtTrn.mjTRN_JOINT)
a.gaintype = mujoco.mjtGain.mjGAIN_FIXED
a.biastype = mujoco.mjtBias.mjBIAS_AFFINE      # motor 는 mjBIAS_NONE (기본값)
a.gainprm[0] = ...;  a.biasprm[:3] = [...]
```
`ctrllimited`, `forcelimited` 는 `True` 를 넣으면 됩니다.
</details>


In [ ]:
def build_actuated_leg(kind: str = "motor", kp: float = 100.0, kd: float = 20.0, effort_limit: float = 386.0) -> mujoco.MjSpec:
    spec = build_leg_spec()
    for jname in LEG_JOINT_AXES:
        a = spec.add_actuator(name=f"{jname}_Actuator", target=jname, trntype=mujoco.mjtTrn.mjTRN_JOINT)
        a.gaintype = mujoco.mjtGain.mjGAIN_FIXED
        if kind == "motor":
            a.gainprm[0] = 1.0
            a.ctrlrange = [-effort_limit, effort_limit]
            a.ctrllimited = True
        elif kind == "position":
            a.gainprm[0] = kp
            a.biastype = mujoco.mjtBias.mjBIAS_AFFINE
            a.biasprm[:3] = [0.0, -kp, -kd]
            a.forcerange = [-effort_limit, effort_limit]
            a.forcelimited = True
        else:
            raise ValueError(kind)
    return spec


In [ ]:
_mm = build_actuated_leg("motor").compile()
_mp = build_actuated_leg("position", kp=100, kd=20).compile()
check("액추에이터 3개", _mm.nu == 3 and _mp.nu == 3)
check("이름 규칙", _mm.actuator(0).name == "Leg_FL_Joint_01_Actuator")
check_close("motor: ctrlrange", _mm.actuator_ctrlrange, [[-386, 386]] * 3)
check_close("motor: gain 1, bias 0", np.c_[_mm.actuator_gainprm[:, 0], _mm.actuator_biasprm[:, :3]], [[1, 0, 0, 0]] * 3)
check_close("position: gain kp, bias (0, -kp, -kd)", np.c_[_mp.actuator_gainprm[:, 0], _mp.actuator_biasprm[:, :3]], [[100, 0, -100, -20]] * 3)
check_close("position: forcerange", _mp.actuator_forcerange, [[-386, 386]] * 3)


### 먼저 예상해 보기

다리를 $q^* = (0.3, -0.869, 0.994)$ 로 PD 제어합니다(kp=100, kd=20). 충분히 기다리면 관절각은 **정확히** $q^*$ 가 될까요? 안 된다면 오차는 얼마일까요?

<details><summary>힌트</summary>

정지 상태에서는 PD 토크가 중력 토크와 같아야 합니다: $k_p (q^* - q) = g(q)$. MuJoCo 는 중력·코리올리 항을 `d.qfrc_bias` 에 넣어 줍니다.
</details>

### 문제 6-3. 두 방식으로 같은 제어를

`motor` 모델은 매 스텝 `pd_torque` 로 `ctrl` 을 계산해 넣고, `position` 모델은 `ctrl = q_des` 만 넣습니다. 둘 다 `seconds` 동안 돌려 관절각 궤적 `(스텝 수, 3)` 을 돌려줍니다.


In [ ]:
def run_leg_pd(kind: str, q_des, kp=100.0, kd=20.0, seconds=2.0, dt=0.001):
    """(궤적 (T, 3), model, data) 를 돌려준다."""
    spec = build_actuated_leg(kind, kp=kp, kd=kd)
    spec.option.timestep = dt
    m = spec.compile()
    d = mujoco.MjData(m)
    traj = []
    for _ in range(round(seconds / dt)):
        if kind == "motor":
            d.ctrl[:] = pd_torque(d.qpos, d.qvel, q_des, kp, kd, torque_limit=386.0)
        else:
            d.ctrl[:] = q_des
        mujoco.mj_step(m, d)
        traj.append(d.qpos.copy())
    return np.array(traj), m, d


In [ ]:
_qdes = np.array([0.3, -0.869, 0.994])
_tm, _mm, _dm = run_leg_pd("motor", _qdes)
_tp, _mp, _dp = run_leg_pd("position", _qdes)
check("두 방식의 궤적이 거의 같음 (최대 차이 < 0.01 rad)", np.abs(_tm - _tp).max() < 0.01)
print("   2초 뒤 관절각:", _tm[-1], " 목표:", _qdes)
check("정확히 목표에 가지는 않는다", np.abs(_qdes - _tm[-1]).max() > 0.01)
check_close("정상상태 오차 × kp = 중력 토크 (qfrc_bias)", (_qdes - _tm[-1]) * 100.0, _dm.qfrc_bias, atol=0.05)
print("   중력 토크:", _dm.qfrc_bias, "N·m  →  처짐:", _dm.qfrc_bias / 100.0, "rad")


kp=100 에서 허벅지 관절이 약 0.046 rad 처집니다. 다리 하나의 무게만으로 이 정도이니, 57 kg 로봇 전체를 받치면 처짐은 훨씬 큽니다(Part 9 에서 직접 봅니다).

World 는 이 처짐을 없애지 않고 **이용** 합니다. 정책의 액션이 0 일 때의 PD 목표(`action_offset`)를 일부러 "실제로 서 있는 자세" 와 다르게 잡아서, 그 차이 × kp 가 딱 지지 토크가 되게 합니다. `diden_common/robots/diden_spider/constants.py` 의 `DEFAULT_JOINT_ANGLES_XML` 주석이 이 내용입니다("these angles are back-solved from the wanted torque and kp").

### 내장 position 액추에이터를 쓰는 이유

두 방식의 궤적이 같다면 왜 World 는 굳이 모터를 지우고 position 액추에이터를 넣을까요? 차이는 **적분** 에 있습니다. `implicitfast` 적분기는 액추에이터 힘의 속도 의존 항($-k_v \dot q$)을 암시적으로 적분합니다. 파이썬에서 계산해 `ctrl` 로 넣은 토크는 그 스텝 동안 상수로 취급되므로 명시적 적분이 됩니다. 감쇠가 크거나 스텝이 길면 명시적 쪽이 먼저 불안정해집니다.


In [ ]:
# 부숴 보기: kd 를 키우고 dt 를 늘려 본다. 마지막 0.2초 동안의 관절 속도 크기를 비교.
for _kd, _dt in [(20, 0.001), (200, 0.002), (500, 0.002), (500, 0.004)]:
    _row = []
    for _kind in ("motor", "position"):
        _t, _, _ = run_leg_pd(_kind, _qdes, kd=_kd, dt=_dt, seconds=1.0)
        _n = round(0.2 / _dt)
        _row.append(np.abs(np.diff(_t[-_n:], axis=0) / _dt).max())
    print(f"kd={_kd:4d} dt={_dt}:  직접 계산(motor) |q̇|max = {_row[0]:8.3f}   내장(position) |q̇|max = {_row[1]:8.3f}  rad/s")


`kd·dt / (관절 관성)` 이 2 에 가까워지면 직접 계산 쪽이 떨기 시작합니다. 여기서 `armature` 의 역할도 보입니다. 종아리 링크 자체의 관성은 아주 작아서, 로터 관성(0.2358 kg·m²)이 없으면 같은 kd 에서 훨씬 빨리 불안정해집니다. World 가 수동 발목 관절에 `ANKLE_ARMATURE = 0.005` 를 일부러 넣는 이유도 같습니다(`constants.py` 주석: "XML ships 0, leaving the ~15 g foot dofs near-singular").

**정리**

- 액추에이터는 `gain·ctrl + bias(q, q̇)` 입니다. motor 와 position 은 계수만 다릅니다.
- PD 제어에는 정상상태 오차가 있고, 그 크기는 `중력 토크 / kp` 입니다.
- 같은 PD 라도 "어디서 계산하느냐" 가 다르면 결과가 달라집니다. World(내장), Core 시뮬레이터(직접 계산), 실제 로봇(드라이버)이 서로 다르다는 점은 섹터 4(Core RL 제어 스택)에서 다시 다룹니다.


## Part 7 — site 와 센서: IMU

**site** 는 body 에 붙은 "표시점" 입니다. 질량도 충돌도 없고, 위치와 방향만 있습니다. 센서를 달 자리, 힘을 걸 자리, 발끝처럼 추적하고 싶은 점에 씁니다. Spider 의 `robot.xml` 에는 IMU 자리(`Torso_IMU_site_IAM`), 발끝(`Foot_*_Tip_site`), 자석 힘 작용점(`*_EPMForcePoint_*_site`) 등이 site 로 들어 있습니다.

**sensor** 는 매 스텝 값을 계산해 `d.sensordata` 라는 긴 벡터 하나에 이어 붙여 줍니다. 센서 `i` 의 값은 `d.sensordata[m.sensor_adr[i] : m.sensor_adr[i] + m.sensor_dim[i]]` 에 있습니다.

| MJCF | 값 | 차원 |
|---|---|---|
| `<accelerometer site=..>` | site 좌표계에서 본 **비력**(specific force) = 가속도 − 중력 | 3 |
| `<gyro site=..>` | site 좌표계에서 본 각속도 | 3 |
| `<framequat objtype="site" objname=..>` | site 의 세계 좌표계 자세 | 4 |
| `<jointpos joint=..>`, `<jointvel joint=..>` | 관절 각도, 속도 | 1 |

실제 로봇에는 "몸통의 세계 좌표계 자세" 를 직접 알려 주는 센서가 없습니다. 있는 것은 가속도계, 자이로, 관절 엔코더뿐이고, 자세·속도는 **추정** 해야 합니다(Core 의 `engine/estimator`, 섹터 5). 그래서 정책의 관측도 이 센서들로 만들 수 있는 양으로 제한합니다. World 의 actor 관측 첫 두 항이 `imu_ang_vel` 과 `projected_gravity` 인 이유입니다.

### 문제 7-1. IMU 가 달린 상자

`MjSpec` 으로 다음을 만듭니다.

- body `box`: 위치 `(0, 0, 0.5)`, freejoint `box_free`, 상자 geom 하나. 그리고 `gravcomp = 1`.
- site `imu`: body 원점
- 센서 3개: `accelerometer`(가속도계), `gyro`(자이로), `orientation`(framequat) — 모두 site `imu` 에

`gravcomp=1` 은 그 body 에 중력과 정확히 반대인 힘을 걸어 줍니다. 누군가 손으로 들고 있는 것과 같아서, 어떤 자세로 돌려 놓아도 그 자리에 가만히 있습니다. (세계에 용접된 body 는 MuJoCo 가 "움직이지 않는 것" 으로 취급해 가속도계가 0 을 읽으므로, 이 실험에는 쓸 수 없습니다.)

<details><summary>힌트</summary>

```python
box = spec.worldbody.add_body(name="box", pos=[0, 0, 0.5], gravcomp=1)
spec.add_sensor(name="accelerometer", type=mujoco.mjtSensor.mjSENS_ACCELEROMETER,
                objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
```
자이로는 `mjSENS_GYRO`, 자세는 `mjSENS_FRAMEQUAT` 입니다.
</details>


In [ ]:
def build_imu_box_spec() -> mujoco.MjSpec:
    spec = mujoco.MjSpec()
    spec.worldbody.add_light(pos=[0, 0, 3])
    box = spec.worldbody.add_body(name="box", pos=[0, 0, 0.5], gravcomp=1)
    box.add_freejoint(name="box_free")
    box.add_geom(name="box_geom", type=mujoco.mjtGeom.mjGEOM_BOX, size=[0.2, 0.1, 0.05])
    box.add_site(name="imu", pos=[0, 0, 0])
    for name, stype in (("accelerometer", mujoco.mjtSensor.mjSENS_ACCELEROMETER),
                        ("gyro", mujoco.mjtSensor.mjSENS_GYRO),
                        ("orientation", mujoco.mjtSensor.mjSENS_FRAMEQUAT)):
        spec.add_sensor(name=name, type=stype, objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
    return spec


def read_sensor(m, d, name: str) -> np.ndarray:
    """센서 이름으로 d.sensordata 의 해당 구간을 복사해 돌려준다."""
    s = m.sensor(name)
    adr, dim = int(s.adr[0]), int(s.dim[0])
    return d.sensordata[adr:adr + dim].copy()


In [ ]:
m_imu = build_imu_box_spec().compile()
d_imu = mujoco.MjData(m_imu)
mujoco.mj_forward(m_imu, d_imu)
check("센서 3개, sensordata 길이 3+3+4", m_imu.nsensor == 3 and m_imu.nsensordata == 10)
check_close("자세: 기울이지 않았으므로 단위 쿼터니언", read_sensor(m_imu, d_imu, "orientation"), [1, 0, 0, 0])
check_close("자이로: 정지", read_sensor(m_imu, d_imu, "gyro"), [0, 0, 0])
check_close("가속도계: 위쪽으로 9.81", read_sensor(m_imu, d_imu, "accelerometer"), [0, 0, 9.81], atol=1e-9)
for _ in range(100):
    mujoco.mj_step(m_imu, d_imu)
check_close("gravcomp 덕분에 떨어지지 않음", d_imu.qpos[:3], [0, 0, 0.5], atol=1e-9)


### 먼저 예상해 보기

1. 가만히 들고 있는 상자의 가속도계는 `(0, 0, +9.81)` 을 읽습니다. 왜 **위쪽** 일까요?
2. 자유낙하 중인 공에 가속도계를 달면 무엇을 읽을까요?
3. 상자를 x 축으로 30° 기울이면 가속도계 값은 어떻게 변할까요?

### 투영 중력 (projected gravity)

"몸통이 얼마나 기울었는가" 를 정책에 알려 주는 표준 방법이 **몸통 좌표계에서 본 중력 방향** 입니다.

$$g^{B} = R(q)^{\top}\,(0, 0, -1)$$

똑바로 서 있으면 `(0, 0, -1)`, 기울면 x, y 성분이 생깁니다. 쿼터니언 4개를 그대로 넣는 것보다 좋은 이유는 (1) yaw(어느 쪽을 보는가)에 의존하지 않고 (2) 정지 상태의 가속도계로 직접 잴 수 있기 때문입니다.

### 문제 7-2. 투영 중력

Part 2 에서 만든 `quat_rotate_inverse` 를 씁니다.


In [ ]:
def projected_gravity(quat) -> np.ndarray:
    """세계 좌표계 자세 쿼터니언 (w,x,y,z) → body 좌표계에서 본 단위 중력 방향."""
    return quat_rotate_inverse(quat, np.array([0.0, 0.0, -1.0]))


In [ ]:
check_close("똑바로: (0, 0, -1)", projected_gravity([1, 0, 0, 0]), [0, 0, -1])
_rng = np.random.default_rng(2)
for _ in range(4):
    _q = _rng.normal(size=4); _q /= np.linalg.norm(_q)
    set_free_body(m_imu, d_imu, "box_free", pos=[0, 0, 0.5], quat=_q)      # 상자를 임의 자세로 들고 있는다
    _acc = read_sensor(m_imu, d_imu, "accelerometer")
    check_close("정지 상태: 투영 중력 = −가속도계 / |가속도계|", projected_gravity(read_sensor(m_imu, d_imu, "orientation")), -_acc / np.linalg.norm(_acc), atol=1e-9)

# 자유낙하: 가속도계는 0 을 읽는다
_spec = mujoco.MjSpec.from_string(drop_xml(5.0))
_spec.body("ball").add_site(name="imu")
_spec.add_sensor(name="accelerometer", type=mujoco.mjtSensor.mjSENS_ACCELEROMETER, objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
_m = _spec.compile(); _d = mujoco.MjData(_m)
for _ in range(50):
    mujoco.mj_step(_m, _d)
check_close("자유낙하 중 가속도계 ≈ 0", read_sensor(_m, _d, "accelerometer"), [0, 0, 0], atol=1e-6)


**정리**

- 가속도계는 가속도가 아니라 **비력** 을 잽니다. 정지 상태에서는 바닥이 받쳐 주는 힘 때문에 위쪽 9.81 을 읽고, 자유낙하에서는 0 입니다. 로봇이 걸을 때는 여기에 몸통의 실제 가속도가 섞이므로, 가속도계만으로 기울기를 알 수 없습니다. 자이로 적분과 섞어 쓰는 필터(Core 의 InEKF, complementary)가 필요한 이유입니다.
- 시뮬레이션에서는 `d.qpos` 의 쿼터니언으로 투영 중력을 "정답" 으로 계산할 수 있습니다. World 는 그렇게 계산한 뒤 **노이즈와 편향을 일부러 섞어** actor 에 줍니다(`projected_gravity`: 기울기 ±0.05 rad, 에피소드 편향 ±0.08 rad). 실제 로봇의 추정 오차를 흉내 내는 것입니다.
- `robot.xml` 의 IMU site 주석을 읽어 보면 장착 위치를 mm 단위로 재서 넣었고, 축 방향은 아직 확정되지 않았다고 적혀 있습니다. site 하나의 `quat` 이 sim-to-real 을 좌우할 수 있습니다.


## Part 8 — `MjSpec` 으로 씬 조립하기: 공 굴리기 씬

실제 씬은 파일 하나로 쓰지 않습니다. 바닥, 로봇, 구조물이 각자 파일로 있고 **실행할 때 조립** 합니다. World 의 조립 코드는 짧습니다(`diden_mjlab/scene/scene.py`).

```python
self._spec = mujoco.MjSpec.from_file(scene.xml)                 # 빈 씬 (조명·시각 설정만)
self._spec.attach(terrain.spec, prefix="", frame=worldbody.add_frame())      # 지형
self._spec.attach(ent.spec, prefix=f"{ent_name}/", frame=frame)             # 로봇·오브젝트마다
model = self._spec.compile()
```

Core 도 같은 일을 C API 로 합니다(`src/engine/simulator/scene_manager.cpp` 의 `mjs_attach`).

`attach` 의 규칙 세 가지:

1. 붙는 쪽(child)의 모든 이름 앞에 `prefix` 가 붙습니다. `sphere` → `ball/sphere`. 로봇을 두 대 넣어도 이름이 겹치지 않습니다.
2. child 는 `frame` 의 위치·자세에 놓입니다.
3. `<option>` 은 **붙이는 쪽(parent)의 것만 남습니다.** child 파일에 적힌 timestep, 적분기 설정은 버려집니다.

붙인 뒤에도 spec 은 고칠 수 있습니다. `spec.body("ball/sphere")` 로 body 를 찾아 관절을 달고, `body.geoms` 를 돌며 속성을 바꿉니다.

### mocap body

섹터 1 에서는 "목적지" 를 화면에 보여 줘야 합니다. 물리에는 참여하지 않고 **코드가 위치를 직접 정하는** 물체가 mocap body 입니다. `add_body(name=.., mocap=True)` 로 만들고, 위치는 `d.mocap_pos[i]` 에 써넣습니다. `qpos` 에는 들어가지 않습니다.

### 문제 8-1. 공 굴리기 씬

섹터 1 에서 그대로 쓸 씬입니다. `DIDEN_Asset/object/sphere.xml` 을 불러와 **굴러다니는 공** 으로 바꿉니다.

| 항목 | 값 |
|---|---|
| `spec.option.timestep` | 인자 `timestep` (기본 0.005) |
| 바닥 | geom `floor`, plane, `size=[0, 0, 0.05]` (0 이면 무한 평면), `friction=[floor_friction, 0.005, 0.0001]` |
| 공 | `sphere.xml` 을 `prefix="ball/"` 로 attach. frame 위치 `(0, 0, 0.2)` (반지름 0.2 → 바닥에 딱 닿음) |
| 공의 관절 | `ball/sphere` 에 freejoint, 이름 `ball/root` |
| 공의 collision geom | 질량이 `ball_mass` 가 되는 밀도, `friction=[ball_friction, 0.005, 0.0001]` |
| 목적지 | mocap body `goal`, 위치 `(1, 0, 0)`. geom `goal_marker`: cylinder, `size=[0.25, 0.005, 0]`, 반투명 초록, `contype=0`, `conaffinity=0` |
| 조명 | 하나 |

collision geom 과 visual geom 은 `contype` 으로 구분할 수 있습니다(visual 은 0).


In [ ]:
BALL_RADIUS = 0.2


def build_ball_scene(ball_mass: float = 1.0, ball_friction: float = 1.0, floor_friction: float = 1.0,
                     timestep: float = 0.005) -> mujoco.MjSpec:
    spec = mujoco.MjSpec()
    spec.modelname = "ball_scene"
    G = mujoco.mjtGeom
    spec.option.timestep = timestep
    wb = spec.worldbody
    wb.add_light(name="sun", pos=[0, 0, 6], dir=[0, 0, -1], type=mujoco.mjtLightType.mjLIGHT_DIRECTIONAL)
    wb.add_geom(name="floor", type=G.mjGEOM_PLANE, size=[0, 0, 0.05], friction=[floor_friction, 0.005, 0.0001])

    child = mujoco.MjSpec.from_file(str(ASSET / "object" / "sphere.xml"))
    spec.attach(child, prefix="ball/", frame=wb.add_frame(pos=[0, 0, BALL_RADIUS]))
    ball = spec.body("ball/sphere")
    ball.add_freejoint(name="ball/root")
    for g in ball.geoms:
        if g.contype:      # collision geom
            g.density = ball_mass / (4.0 / 3.0 * np.pi * BALL_RADIUS**3)
            g.friction = [ball_friction, 0.005, 0.0001]

    goal = wb.add_body(name="goal", mocap=True, pos=[1, 0, 0])
    goal.add_geom(name="goal_marker", type=G.mjGEOM_CYLINDER, size=[0.25, 0.005, 0], rgba=[0.1, 0.9, 0.2, 0.6],
                  contype=0, conaffinity=0)
    return spec


In [ ]:
_m = build_ball_scene(ball_mass=1.0, ball_friction=0.5, floor_friction=0.3).compile()
check("body: world, ball/sphere, goal", {_m.body(i).name for i in range(_m.nbody)} == {"world", "ball/sphere", "goal"})
check("freejoint ball/root 하나뿐 (nq=7, nv=6)", _m.njnt == 1 and _m.joint("ball/root").type[0] == mujoco.mjtJoint.mjJNT_FREE and _m.nq == 7)
check_close("공 질량", _m.body("ball/sphere").mass, [1.0], atol=1e-9)
check_close("공 관성 2/5 m r²", _m.body("ball/sphere").inertia, [0.016] * 3, atol=1e-9)
check_close("공 마찰", _m.geom("ball/sphere_collision").friction, [0.5, 0.005, 0.0001])
check_close("바닥 마찰", _m.geom("floor").friction, [0.3, 0.005, 0.0001])
check("visual geom 은 그대로 (충돌·질량 없음)", _m.geom("ball/sphere_visual").contype[0] == 0)
check("goal 은 mocap body", _m.nmocap == 1 and _m.body("goal").mocapid[0] == 0)
check("goal_marker 는 충돌 안 함", _m.geom("goal_marker").contype[0] == 0 and _m.geom("goal_marker").conaffinity[0] == 0)
check("timestep", _m.opt.timestep == 0.005)
_d = mujoco.MjData(_m)
for _ in range(200):
    mujoco.mj_step(_m, _d)
check("공이 바닥 위에 놓여 있음", abs(_d.qpos[2] - 0.2) < 2e-3 and _d.ncon == 1)

# 토크로 굴리기: Part 4 의 해석해와 맞는가 (유효 마찰 = max(0.5, 0.3) = 0.5)
_tau = 0.5 * max_torque_without_slip(0.5, 1.0, BALL_RADIUS)
_d.xfrc_applied[_m.body("ball/sphere").id, 4] = _tau
_v0 = _d.qvel[0]
for _ in range(200):
    mujoco.mj_step(_m, _d)
check_close("y 축 토크 → +x 로 가속, a = τ / (7/5·m·r)", (_d.qvel[0] - _v0) / 1.0, predict_ball_accel_from_torque(_tau, 1.0, BALL_RADIUS), rtol=0.01)


### 굴려 보기

아래는 주어진 코드입니다. 바닥에 격자 무늬를 입히고(구르는 것이 보이도록), 목적지를 옮기고, 목적지 방향으로 토크를 걸어 굴립니다. 제어기는 "목적지 쪽으로 일정한 토크" 가 전부입니다.

### 먼저 예상해 보기

이 제어기로 공은 목적지에 **멈출** 수 있을까요? Part 4 의 마지막 "부숴 보기" 를 떠올려 봅니다.


In [ ]:
def add_grid_floor(spec: mujoco.MjSpec):
    """바닥에 체커 무늬 재질을 입힌다 (보기 좋게 하는 용도. 물리는 그대로)."""
    spec.add_texture(name="grid", type=mujoco.mjtTexture.mjTEXTURE_2D, builtin=mujoco.mjtBuiltin.mjBUILTIN_CHECKER,
                     width=256, height=256, rgb1=[0.22, 0.24, 0.28], rgb2=[0.30, 0.32, 0.37])
    mat = spec.add_material(name="grid", texrepeat=[1, 1])
    mat.textures[mujoco.mjtTextureRole.mjTEXROLE_RGB] = "grid"
    spec.geom("floor").material = "grid"
    return spec


def torque_toward(m, d, goal_xy, magnitude: float):
    """공에서 목적지를 향하는 방향 u 로 구르게 하는 세계 좌표계 토크를 xfrc_applied 에 쓴다.
    +x 로 굴리려면 +y 토크, +y 로 굴리려면 −x 토크: τ = magnitude · (ẑ × u)."""
    bid = m.body("ball/sphere").id
    u = np.asarray(goal_xy) - d.xpos[bid][:2]
    u = u / (np.linalg.norm(u) + 1e-9)
    d.xfrc_applied[bid, 3:6] = magnitude * np.array([-u[1], u[0], 0.0])


m_ball = add_grid_floor(build_ball_scene()).compile()
d_ball = mujoco.MjData(m_ball)
goal_xy = np.array([2.0, 1.0])
d_ball.mocap_pos[0] = [*goal_xy, 0.0]
frames = record(m_ball, d_ball, 4.0, camera=make_camera(lookat=(1.0, 0.5, 0.2), distance=5.0, azimuth=120, elevation=-30),
                step_fn=lambda m, d: torque_toward(m, d, goal_xy, 0.5), groups=[0, 1, 2])
media.show_video(frames, fps=30)
print("4초 뒤 공 위치:", d_ball.body("ball/sphere").xpos[:2], " 목적지:", goal_xy, " 속도:", d_ball.qvel[:2])


공은 목적지를 지나쳤다가 되돌아오기를 반복합니다. 멈추려면 속도를 보고 미리 반대 토크를 걸어야 합니다. 그 "언제 얼마나" 를 손으로 짜는 대신 학습시키는 것이 섹터 1 입니다.

**정리**

- 씬 = 빈 spec + `attach(child, prefix, frame)` 여러 번 + 수정 + `compile()`. World 의 `MujocoWarpScene`, Core 의 `scene_manager` 가 하는 일이 이것입니다.
- attach 하면 이름에 prefix 가 붙습니다. World 에서 로봇의 body 는 전부 `robot/...` 입니다. 코드에서 이름으로 body 를 찾을 때 prefix 를 빼먹는 실수가 흔합니다.
- 컴파일 **전** 에 고치는 것(spec: 관절 추가, geom 속성)과 컴파일 **후** 에 고치는 것(model: `m.geom_friction[...] = ...`)을 구분합니다. 구조를 바꾸는 일은 전에만 가능하고, 숫자 하나 바꾸는 일은 후에도 됩니다. 섹터 1 의 도메인 랜덤화는 후자입니다.


## Part 9 — 진짜 로봇: Spider 불러와 세우기

지금까지 만든 것을 모두 써서 `DIDEN_Asset/robot/diden_spider_v2/robot.xml` 을 읽습니다. 이 파일 하나를 World 와 Core 가 **같이** 씁니다(둘 다 `DIDEN_Asset` 을 서브모듈로 갖고 있습니다). 로봇의 물리값이 바뀌면 이 파일 한 곳만 고칩니다.


In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    m_robot = mujoco.MjModel.from_xml_path(str(ASSET / "robot" / "diden_spider_v2" / "robot.xml"))
print(f"body {m_robot.nbody}개 | 관절 {m_robot.njnt}개 | nq={m_robot.nq} nv={m_robot.nv} | 액추에이터 {m_robot.nu}개 | 센서 {m_robot.nsensor}개 | geom {m_robot.ngeom}개")
print(f"전체 질량 {m_robot.body_subtreemass[m_robot.body('base').id]:.2f} kg | timestep {m_robot.opt.timestep} | 키프레임 {[m_robot.key(i).name for i in range(m_robot.nkey)]}")


### 먼저 예상해 보기

`nq = 37` 은 어떻게 나온 숫자일까요? 다리 4개, 다리마다 모터 3개와 수동 발목 3개, 그리고 6축 매니퓰레이터가 있습니다.

### 문제 9-1. 기구학 트리 출력

body 트리를 들여쓰기로 보여 주는 줄 목록을 만듭니다. 각 줄은

```
<깊이 × 공백 2칸><body 이름>            ← 관절이 없는 body
<깊이 × 공백 2칸><body 이름> ← <관절 이름>, <관절 이름>
```

형식이고, `world` 가 깊이 0 입니다. 순서는 body id 순서(= MJCF 에 적힌 순서, 깊이 우선)를 그대로 따릅니다.

필요한 배열: `m.body_parentid[b]`, `m.body_jntadr[b]`(첫 관절 id, 없으면 −1), `m.body_jntnum[b]`.


In [ ]:
def tree_lines(m) -> list[str]:
    lines = []
    depth = np.zeros(m.nbody, dtype=int)
    for b in range(m.nbody):
        if b > 0:
            depth[b] = depth[m.body_parentid[b]] + 1
        joints = [m.joint(m.body_jntadr[b] + k).name for k in range(m.body_jntnum[b])]
        lines.append("  " * depth[b] + m.body(b).name + (" ← " + ", ".join(joints) if joints else ""))
    return lines


In [ ]:
_lines = tree_lines(m_robot)
check("줄 수 = body 수", len(_lines) == m_robot.nbody)
check("world", _lines[0] == "world")
check("base 는 freejoint 'root'", _lines[1] == "  base ← root")
check("Torso 는 base 에 고정", "    Torso" in _lines)
check("엉덩이 관절", "      Leg_FL_Link_01 ← Leg_FL_Joint_01" in _lines)
check("발목 roll 은 깊이 8", "  " * 8 + "Foot_FL_Link_03 ← Foot_FL_Joint_03" in _lines)
print("\n".join(l for l in _lines if "Leg_FL" in l or "Foot_FL" in l or l.strip() in ("world", "Torso") or "base" in l))


`Foot_FL_EPMForcePoint_01~04` 는 발바닥 네 귀퉁이에 있는 질량 0.01 g 짜리 body 입니다. 자석(EPM)이 당기는 힘을 `xfrc_applied` 로 걸 **작용점** 으로 쓰기 위해 body 로 만들어 둔 것입니다(Part 4 에서 본 대로 `xfrc_applied` 는 body 단위입니다). World 의 `diden_domain/magnet`, Core 의 `simulator.cpp` 가 둘 다 이 body 들에 힘을 겁니다.

### 문제 9-2. 구동 관절의 순서

정책이 내는 액션 벡터의 i 번째 칸이 어느 관절인가는 **액추에이터 순서** 로 정해집니다. 이 순서가 World(학습)와 Core(실행)에서 다르면 로봇은 엉뚱한 다리를 움직입니다. 순서를 뽑아 봅니다.

액추에이터 `i` 가 구동하는 관절 id 는 `m.actuator_trnid[i, 0]` 입니다.


In [ ]:
def actuated_joints(m):
    """(관절 이름 리스트, qpos 주소 배열, qvel 주소 배열) — 액추에이터 순서."""
    jids = [int(m.actuator_trnid[i, 0]) for i in range(m.nu)]
    names = [m.joint(j).name for j in jids]
    qadr = np.array([m.jnt_qposadr[j] for j in jids])
    vadr = np.array([m.jnt_dofadr[j] for j in jids])
    return names, qadr, vadr


In [ ]:
_names, _qadr, _vadr = actuated_joints(m_robot)
check("18개 (다리 12 + 팔 6)", len(_names) == 18)
check("다리 순서는 RR → RL → FR → FL", [n.split("_")[1] for n in _names[:12:3]] == ["RR", "RL", "FR", "FL"])
check("팔이 마지막", _names[12:] == [f"Manipulator_Joint_0{i}" for i in range(1, 7)])
check("수동 발목은 포함되지 않음", not any(n.startswith("Foot_") for n in _names))
check("qpos 주소: 다리마다 발목 3칸을 건너뛴다", list(_qadr[:6]) == [7, 8, 9, 13, 14, 15])
check("qvel 주소는 qpos 주소 − 1 (freejoint 가 7칸/6칸)", np.array_equal(_vadr, _qadr - 1))
print("  ", _names[:12])


### 문제 9-3. 로봇을 씬에 넣기

`robot.xml` 을 `MjSpec` 으로 불러 바닥이 있는 씬에 붙입니다. World 의 `diden_mjlab/robots/diden_spider/spec.py` 가 하는 일의 축소판입니다.

1. `MjSpec.from_file(robot.xml)` 로 불러온다.
2. `robot.xml` 안에는 단독으로 열어 볼 때를 위한 보이지 않는 바닥 `__mjv_default_floor` 가 있다. 지운다: `spec.delete(spec.geom("__mjv_default_floor"))`. (World 의 `_strip_standalone_floor`)
3. 새 씬 spec 을 만든다: 라디안, `timestep = 0.001`, `implicitfast`, 조명, plane 바닥 `floor`. 로봇이 어두운 색이라 바닥은 밝게 칠해 두면 보기 좋습니다(예: `rgba=[0.75, 0.75, 0.78, 1]`).
4. `prefix="spider/"` 로 attach 한다.

World 는 여기에 더해 키프레임 삭제, 모터 삭제 후 position 액추에이터 주입, 매니퓰레이터 용접 등을 합니다. 우리는 모터와 키프레임을 그대로 둡니다.


In [ ]:
def build_spider_scene() -> mujoco.MjSpec:
    robot = mujoco.MjSpec.from_file(str(ASSET / "robot" / "diden_spider_v2" / "robot.xml"))
    robot.delete(robot.geom("__mjv_default_floor"))

    scene = mujoco.MjSpec()
    scene.compiler.degree = False
    scene.option.timestep = 0.001
    scene.option.integrator = mujoco.mjtIntegrator.mjINT_IMPLICITFAST
    scene.worldbody.add_light(name="sun", pos=[0, 0, 6], dir=[0, 0, -1], type=mujoco.mjtLightType.mjLIGHT_DIRECTIONAL)
    scene.worldbody.add_geom(name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE, size=[0, 0, 0.05], rgba=[0.75, 0.75, 0.78, 1])
    scene.attach(robot, prefix="spider/", frame=scene.worldbody.add_frame())
    return scene


In [ ]:
with warnings.catch_warnings(record=True) as _w:
    warnings.simplefilter("always")
    m_spider = build_spider_scene().compile()
print("attach 가 낸 경고:\n  " + "\n  ".join(str(_w[0].message).splitlines()) if _w else "(경고 없음)")
check("이름에 prefix", m_spider.body("spider/Torso").id > 0 and m_spider.joint("spider/root").type[0] == mujoco.mjtJoint.mjJNT_FREE)
check("로봇의 숨은 바닥은 지워짐", "spider/__mjv_default_floor" not in [m_spider.geom(i).name for i in range(m_spider.ngeom)])
check("우리 바닥이 있음", m_spider.geom("floor").type[0] == mujoco.mjtGeom.mjGEOM_PLANE)
check("자유도는 그대로 (nq=37, nu=18)", m_spider.nq == 37 and m_spider.nu == 18)
check("씬의 option 이 이김: timestep 0.001, noslip 0", m_spider.opt.timestep == 0.001 and m_spider.opt.noslip_iterations == 0)
check("키프레임도 prefix 가 붙어 따라옴", m_spider.key("spider/stand_pose").id >= 0)


경고를 읽어 봅니다. `robot.xml` 에 적힌 `timestep=0.0005`, `noslip_iterations=20`, `gravity=-9.80665` 가 **버려졌다** 는 내용입니다. attach 에서는 붙이는 쪽의 `<option>` 만 남기 때문입니다. "로봇 파일에 적힌 물리 설정" 과 "실제로 도는 물리 설정" 은 다를 수 있고, World 에서 실제 값은 sim profile(`diden_configs/sim_profiles/mujoco_warp/diden_spider.py`)이 정합니다.

### 문제 9-4. 세우기

키프레임 `stand_pose` 의 관절각을 목표로 PD 제어해 로봇을 세웁니다.

1. `mujoco.mj_resetDataKeyframe(m, d, key_id)` 로 키프레임 상태로 만든다.
2. 키프레임의 몸통 높이는 바닥 기준이 아니므로, 발바닥이 바닥에 닿도록 몸통을 옮긴다 → 주어진 `place_on_floor` 사용.
3. 매 스텝 구동 관절 18개에 `pd_torque` 를 계산해 `d.ctrl` 에 넣는다. 목표는 키프레임의 관절각, 토크 한계는 `m.actuator_ctrlrange`.
4. `seconds` 뒤 `(몸통 높이, 몸통 z축의 세계 z 성분)` 을 돌려준다. 두 번째 값은 똑바로 서 있으면 1, 옆으로 누우면 0 입니다 — `d.body("spider/Torso").xmat` 의 마지막 원소.

### 먼저 예상해 보기

World 의 기본 게인은 kp=100, kd=20 입니다. 이 게인으로 `stand_pose` 를 목표로 주면 57 kg 로봇이 설 수 있을까요? Part 6 의 "처짐 = 중력 토크 / kp" 를 떠올려 봅니다.


In [ ]:
def place_on_floor(m, d, clearance: float = 0.002):
    """(주어진 코드) 가장 낮은 발바닥 상자가 바닥에서 clearance 만큼 뜨도록 몸통을 위아래로 옮긴다."""
    mujoco.mj_forward(m, d)
    soles = [i for i in range(m.ngeom) if m.geom(i).name.endswith("Link_03_collision") and "Foot" in m.geom(i).name]
    lowest = min(d.geom_xpos[i][2] - m.geom_size[i][2] for i in soles)
    d.qpos[2] += clearance - lowest
    mujoco.mj_forward(m, d)


def stand(m, kp: float, kd: float, seconds: float = 4.0, key: str = "spider/stand_pose"):
    """PD 로 키프레임 자세를 유지시키고 (몸통 높이, 몸통 up 성분, data) 를 돌려준다."""
    d = mujoco.MjData(m)
    kid = m.key(key).id
    mujoco.mj_resetDataKeyframe(m, d, kid)
    place_on_floor(m, d)
    _, qadr, vadr = actuated_joints(m)
    q_des = m.key_qpos[kid][qadr].copy()
    limit = m.actuator_ctrlrange[:, 1]
    for _ in range(round(seconds / m.opt.timestep)):
        d.ctrl[:] = pd_torque(d.qpos[qadr], d.qvel[vadr], q_des, kp, kd, torque_limit=limit)
        mujoco.mj_step(m, d)
    torso = d.body("spider/Torso")
    return float(torso.xpos[2]), float(torso.xmat[8]), d


In [ ]:
_z, _up, d_spider = stand(m_spider, kp=400.0, kd=20.0)
print(f"   kp=400: 몸통 높이 {_z:.3f} m, up {_up:.3f}")
check("kp=400 이면 선다", _up > 0.99 and _z > 0.4)
_names, _qadr, _ = actuated_joints(m_spider)
_err = m_spider.key_qpos[m_spider.key("spider/stand_pose").id][_qadr] - d_spider.qpos[_qadr]
print("   다리 관절의 정상상태 오차 [rad]:", _err[:12].round(3))
check("서 있을 때도 처짐은 남는다 (허벅지 관절 > 0.01 rad)", np.abs(_err[1:12:3]).min() > 0.01)
media.show_image(render(m_spider, d_spider, make_camera(lookat=(0, 0, 0.35), distance=2.8, azimuth=140, elevation=-20), groups=[0, 1, 2]))

_z, _up, _ = stand(m_spider, kp=100.0, kd=20.0)
print(f"   kp=100: 몸통 높이 {_z:.3f} m, up {_up:.3f}")
check("kp=100 으로 같은 목표를 주면 주저앉는다 — 예상과 같았나요?", _up < 0.9)


kp=100 에서는 처짐이 커서 다리가 벌어지고, 벌어질수록 필요한 토크가 더 커져서 무너집니다. 그런데 World 는 kp=100 으로 이 로봇을 걷게 합니다. 가능한 이유가 Part 6 끝에서 말한 `action_offset` 입니다. PD 목표를 "서 있는 자세" 가 아니라 "처진 결과가 서 있는 자세가 되는 각도" 로 잡고, 나머지는 정책이 매 스텝 목표를 조정해 메웁니다. Core 의 NMPC 는 다른 길을 갑니다. 지지 다리는 게인을 낮추고(stance kp=0, kd=100) 필요한 힘을 피드포워드 토크 $\tau = -J^\top f$ 로 직접 넣습니다(`config/diden_spider/nmpc_ctrl_config.yaml`). 같은 모터를 쓰는 두 가지 철학입니다.

### 마무리: 로봇과 공을 한 씬에


In [ ]:
scene = add_grid_floor(build_spider_scene())
sph = mujoco.MjSpec.from_file(str(ASSET / "object" / "sphere.xml"))
scene.attach(sph, prefix="ball/", frame=scene.worldbody.add_frame(pos=[0.9, 0.0, 0.6]))
ball = scene.body("ball/sphere")
ball.add_freejoint(name="ball/root")
for g in ball.geoms:
    if g.contype:
        g.density = sphere_density(1.0, BALL_RADIUS)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    m_both = scene.compile()
print("nq =", m_both.nq, "(로봇 37 + 공 7)")
_, _, d_both = stand(m_both, kp=400.0, kd=20.0, seconds=2.0)
media.show_image(render(m_both, d_both, make_camera(lookat=(0.3, 0, 0.3), distance=3.2, azimuth=140, elevation=-20), groups=[0, 1, 2]))


## 섹터 0 을 마치며

여기까지 만든 것:

- MJCF 와 `MjSpec` 두 방식으로 모델을 쓰고, `MjModel` / `MjData` 에서 상태를 읽고 쓰는 법
- 질량·충돌·마찰이 geom 에서 오고, default class 와 include 로 공유된다는 것
- 관절 트리와 정기구학·야코비안, 액추에이터와 PD, 센서
- 여러 파일을 한 씬으로 조립하는 법, 그리고 조립 과정에서 로봇 파일의 물리 설정이 덮어씌워진다는 것

**스스로 답해 볼 질문** (`study/QUESTION_TREE.md` 의 Q0 가지)

1. `robot.xml` 의 다리 collision geom 이 메시가 아니라 원기둥·상자인 이유는?
2. Spider 의 geom 기본값이 `condim="4"` 인 이유는? `condim="3"` 으로 바꾸면 벽에 붙은 로봇에 무슨 일이 생길까?
3. World 는 왜 `robot.xml` 의 `<motor>` 를 지우고 position 액추에이터를 넣을까? Core 시뮬레이터는 왜 그대로 둘까?
4. `armature` 를 0 으로 만들면 시뮬레이션에 무슨 일이 생길까? 실제 로봇과 무엇이 달라질까?
5. 같은 `robot.xml` 을 쓰는데도 World 와 Core 의 시뮬레이션 결과가 다를 수 있는 이유를 세 가지 이상 들어 보면?

**부숴 보기**

1. `build_spider_scene` 에서 `scene.compiler.degree = False` 를 지우면? (에러가 나는지, 조용히 틀리는지)
2. `stand` 에서 `place_on_floor` 를 빼면 로봇은 어디서 시작하나? 그래도 서는가?
3. 마지막 씬에서 공을 로봇 바로 위(`pos=[0, 0, 1.5]`)에서 떨어뜨리면 로봇은 버티는가? 공 질량을 20 kg 으로 올리면?

다음은 **섹터 1 — 공 굴리기 환경** 입니다. Part 8 의 씬을 GPU 에 4096 개 올리고, World 와 같은 구조(매니저 기반 MDP)의 환경과 PPO 를 직접 만들어 공이 목적지에 가서 멈추도록 학습시킵니다.
